# MAGNET: Multi-scale Attention-based Graph Network with Ensemble Transfer Learning
# for Breast Cancer Histopathology Classification


## 1. Environment Setup

In [1]:
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg, mod in [('timm','timm'), ('statsmodels','statsmodels'),
                  ('scipy','scipy'), ('albumentations','albumentations')]:
    try:
        __import__(mod)
    except ImportError:
        pip_install(pkg)

print('All packages ready.')


All packages ready.


In [2]:
import os, sys, re, random, json, time, copy, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from torchvision import transforms
import cv2
import scipy.stats as st
from statsmodels.stats.contingency_tables import mcnemar as mcnemar_test_fn
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve, classification_report
)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import timm
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import WeightedRandomSampler

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if device.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1024**2:.0f} MB')

for d in ['results','checkpoints','plots']:
    os.makedirs(d, exist_ok=True)
print('Setup complete.')

# ── CPU-safe autocast helper ───────────────────────────────────────────────
import contextlib

def autocast_ctx():
    """
    Returns torch.amp.autocast for CUDA, or a no-op context on CPU.
    Prevents 'autocast not supported on CPU' errors if GPU is unavailable.
    """
    if device.type == 'cuda':
        return torch.amp.autocast(device_type='cuda')
    return contextlib.nullcontext()

Device : cuda
GPU    : NVIDIA GeForce GTX 1650
VRAM   : 4096 MB
Setup complete.


## 2. Configuration

In [3]:
DATA_ROOT = 'data/BreaKHis_v1/BreaKHis_v1/histology_slides/breast'

# ── Hardware (GTX 1650 4 GB) ──────────────────────────────────────────────
BATCH_SIZE         = 4
NUM_WORKERS        = 0
IMG_SIZE           = 224
IMG_SIZE_WARMUP    = 160    # Phase 3.1: progressive resizing warm-up
FEAT_DIM           = 512
MAGNIFICATIONS     = ['40X', '100X', '200X', '400X']
ACCUM_STEPS        = 4      # gradient accumulation → effective batch = 16
# ⚠ VRAM note: MIL processes BATCH_SIZE × IMAGES_PER_MAG images per forward pass.
# With B=4, N=5: 20 imgs × 4 mags = 80 images at once. If OOM → set IMAGES_PER_MAG=3.

# ── Training ─────────────────────────────────────────────────────────────
EPOCHS             = 60
WARMUP_EPOCHS      = 5      # train at 160px then switch to 224px
LR                 = 5e-4   # heads — safe for pretrained models
LR_BACKBONE        = 5e-5   # backbones — conservative for pretrained weights
WEIGHT_DECAY       = 1e-4
EARLY_STOP         = 15
LABEL_SMOOTHING    = 0.1
FOCAL_GAMMA        = 2.0    # focal loss gamma
IMAGES_PER_MAG     = 3      # max images sampled per patient per mag per epoch

# ── CV ───────────────────────────────────────────────────────────────────
N_FOLDS            = 5
KF_EPOCHS          = 40
KF_PATIENCE        = 10

# ── Upgraded backbones (stronger but still fit 4 GB with AMP) ────────────
BACKBONES = {
    '40X' : 'efficientnet_b2',        # replaces mobilenetv3_large — much stronger
    '100X': 'efficientnet_b1',        # replaces efficientnet_b0
    '200X': 'resnet50',               # replaces resnet34
    '400X': 'efficientnet_b0',        # replaces mobilenetv3_small
}
BACKBONE_OUT_DIM = {
    '40X' : 1408,   # efficientnet_b2
    '100X': 1280,   # efficientnet_b1
    '200X': 2048,   # resnet50
    '400X': 1280,   # efficientnet_b0
}

def _safe_name(s):
    return re.sub(r'[^\w\-]', '_', s)

print('Config loaded.')
print(f'Effective batch size: {BATCH_SIZE * ACCUM_STEPS} (via gradient accumulation)')


Config loaded.
Effective batch size: 16 (via gradient accumulation)


## 3. Data Preparation

### 3.1 Stain Normalisation (Macenko, simplified)

In [4]:
def macenko_norm(img_pil: Image.Image) -> Image.Image:
    """
    OD-space colour normalisation (simplified Macenko).
    For exact Macenko: pip install staintools
    """
    img = np.array(img_pil, dtype=np.float32)
    img = np.clip(img, 1, 255)
    OD  = -np.log(img / 255.0)
    for c in range(3):
        m = OD[..., c].mean()
        s = OD[..., c].std() + 1e-6
        OD[..., c] = (OD[..., c] - m) / s * 0.3 + 0.4
    img_norm = np.exp(-OD) * 255.0
    return Image.fromarray(np.clip(img_norm, 0, 255).astype(np.uint8))

print('Stain normaliser defined.')

Stain normaliser defined.


### 3.2 Patient Index + Stratified Splits

In [5]:
BENIGN_SUBS    = ['adenosis','fibroadenoma','phyllodes_tumor','tubular_adenoma']
MALIGNANT_SUBS = ['ductal_carcinoma','lobular_carcinoma','mucinous_carcinoma','papillary_carcinoma']
ALL_SUBS       = BENIGN_SUBS + MALIGNANT_SUBS
SUB2IDX        = {s: i for i, s in enumerate(ALL_SUBS)}

def collect_patient_info(root_dir):
    """
    Supports both BreakHis folder layouts:
      Layout A (common): breast/benign/SOB/adenosis/<patient>/<mag>/*.png
      Layout B (some downloads): breast/benign/adenosis/<patient>/<mag>/*.png
    """
    patients = {}
    for cls_name, binary, subs in [
            ('benign', 0, BENIGN_SUBS), ('malignant', 1, MALIGNANT_SUBS)]:
        cls_path = os.path.join(root_dir, cls_name)
        if not os.path.isdir(cls_path):
            continue
        # Try SOB subdir first (Layout A), then direct (Layout B)
        sob_path = os.path.join(cls_path, 'SOB')
        search_root = sob_path if os.path.isdir(sob_path) else cls_path

        for sub in subs:
            sub_path = os.path.join(search_root, sub)
            if not os.path.isdir(sub_path):
                continue
            for pat in os.listdir(sub_path):
                pat_path = os.path.join(sub_path, pat)
                if not os.path.isdir(pat_path):
                    continue
                if pat not in patients:
                    patients[pat] = {
                        'binary' : binary,
                        'subtype': SUB2IDX[sub],
                        'paths'  : {m: [] for m in MAGNIFICATIONS}
                    }
                for mag in MAGNIFICATIONS:
                    mag_path = os.path.join(pat_path, mag)
                    if os.path.isdir(mag_path):
                        for fname in os.listdir(mag_path):
                            if fname.lower().endswith('.png'):
                                patients[pat]['paths'][mag].append(
                                    os.path.join(mag_path, fname))
    return patients


# ── Safety: fail loudly if dataset not found ──────────────────────────────────
if not os.path.isdir(DATA_ROOT):
    raise FileNotFoundError(
        f'BreakHis dataset not found at:\n  {DATA_ROOT}\n'
        'Download from: https://web.inf.ufpr.br/vri/databases/'
        'breast-cancer-histopathological-database-breakhis/\n'
        'Then update DATA_ROOT in the config cell.'
    )

patient_info = collect_patient_info(DATA_ROOT)

if len(patient_info) == 0:
    raise RuntimeError(
        f'No patients found under {DATA_ROOT}.\n'
        'Check that the folder structure matches:\n'
        '  breast/benign/SOB/<subtype>/<patient_id>/<magnification>/*.png\n'
        'or:\n'
        '  breast/benign/<subtype>/<patient_id>/<magnification>/*.png'
    )

all_patients = list(patient_info.keys())
pat_labels   = [patient_info[p]['binary'] for p in all_patients]

# ── Dataset statistics ────────────────────────────────────────────────────────
n_benign    = sum(1 for p in all_patients if patient_info[p]['binary'] == 0)
n_malignant = len(all_patients) - n_benign
total_imgs  = sum(len(v) for p in all_patients
                  for v in patient_info[p]['paths'].values())

print(f'Total patients : {len(all_patients)}')
print(f'  Benign       : {n_benign}  ({100*n_benign/len(all_patients):.1f}%)')
print(f'  Malignant    : {n_malignant}  ({100*n_malignant/len(all_patients):.1f}%)')
print(f'Total images   : {total_imgs}')

from collections import Counter
sub_counts = Counter(patient_info[p]['subtype'] for p in all_patients)
print('\nSubtype distribution:')
for idx, name in enumerate(ALL_SUBS):
    print(f'  {name:<25}: {sub_counts.get(idx, 0)} patients')

# Check all patients have images at all magnifications
missing = [(p, mag) for p in all_patients for mag in MAGNIFICATIONS
           if not patient_info[p]['paths'][mag]]
if missing:
    print(f'\n⚠ {len(missing)} patient-magnification combinations have no images:')
    for p, m in missing[:5]:
        print(f'  {p} @ {m}')
    if len(missing) > 5:
        print(f'  ... and {len(missing)-5} more')
else:
    print('\n✓ All patients have images at all 4 magnifications.')

# ── Fixed 70/15/15 patient split (for ablation / interpretability cells) ──────
random.seed(42); random.shuffle(all_patients)
n = len(all_patients)
train_patients = all_patients[:int(0.70 * n)]
val_patients   = all_patients[int(0.70 * n):int(0.85 * n)]
test_patients  = all_patients[int(0.85 * n):]
print(f'\nFixed split — Train:{len(train_patients)} Val:{len(val_patients)} Test:{len(test_patients)}')
print('(5-fold CV in Section 7 uses all patients — recommended for publication)')


Total patients : 82
  Benign       : 24  (29.3%)
  Malignant    : 58  (70.7%)
Total images   : 7909

Subtype distribution:
  adenosis                 : 4 patients
  fibroadenoma             : 10 patients
  phyllodes_tumor          : 3 patients
  tubular_adenoma          : 7 patients
  ductal_carcinoma         : 38 patients
  lobular_carcinoma        : 5 patients
  mucinous_carcinoma       : 9 patients
  papillary_carcinoma      : 6 patients

✓ All patients have images at all 4 magnifications.

Fixed split — Train:57 Val:12 Test:13
(5-fold CV in Section 7 uses all patients — recommended for publication)


### 3.3 Dataset & DataLoaders

In [6]:
import math

from torchvision import transforms
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ── Albumentations pipelines ──────────────────────────────────────────────────
albu_train = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=180, p=0.7),
    A.OneOf([
        A.ElasticTransform(alpha=120, sigma=120*0.05, p=0.5),
        A.GridDistortion(p=0.5),
    ], p=0.3),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.8),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=0.5),
    A.GaussNoise(var_limit=(5.0, 30.0), p=0.3),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2(),
])
albu_val = A.Compose([
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2(),
])

# TTA transforms: same image, different geometric views
TTA_TRANSFORMS = [
    A.Compose([A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), ToTensorV2()]),
    A.Compose([A.HorizontalFlip(p=1.0), A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), ToTensorV2()]),
    A.Compose([A.VerticalFlip(p=1.0),   A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), ToTensorV2()]),
    A.Compose([A.RandomRotate90(p=1.0), A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), ToTensorV2()]),
    A.Compose([A.Transpose(p=1.0),      A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), ToTensorV2()]),
]

IMG_MEAN = [0.485, 0.456, 0.406]
IMG_STD  = [0.229, 0.224, 0.225]
train_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(IMG_MEAN, IMG_STD)])
val_tf   = transforms.Compose([transforms.ToTensor(), transforms.Normalize(IMG_MEAN, IMG_STD)])


class MultiMagDataset(Dataset):
    """
    Option B (MIL): returns a LIST of tensors per magnification.
    Each image is loaded and augmented independently; the backbone
    sees each image separately and features are aggregated AFTER
    the backbone — preserving per-image discriminative detail.

    Returns:
        imgs_per_mag: list of 4 lists, each containing n_per_mag tensors [3,H,W]
        binary_label: int
        subtype_label: int
    """
    def __init__(self, patient_ids, patient_info, augment=False,
                 img_size=224, n_per_mag=IMAGES_PER_MAG):
        self.patient_ids  = patient_ids
        self.patient_info = patient_info
        self.augment      = augment
        self.img_size     = img_size
        self.n_per_mag    = n_per_mag
        self.pipeline     = albu_train if augment else albu_val

    def __len__(self): return len(self.patient_ids)

    def _load(self, path):
        img = Image.open(path).convert('RGB')
        img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        img = macenko_norm(img)
        arr = np.array(img, dtype=np.uint8)
        return self.pipeline(image=arr)['image']   # [3,H,W] float32

    def __getitem__(self, idx):
        pid  = self.patient_ids[idx]
        info = self.patient_info[pid]
        rng  = np.random.default_rng()
        imgs_per_mag = []
        for mag in MAGNIFICATIONS:
            paths = info['paths'][mag]
            if not paths:
                # Pad with zeros if mag missing
                imgs_per_mag.append(
                    [torch.zeros(3, self.img_size, self.img_size)] * self.n_per_mag)
                continue
            n      = min(self.n_per_mag, len(paths))
            chosen = rng.choice(len(paths), size=n, replace=False)
            tensors = [self._load(paths[c]) for c in chosen]
            # Pad to n_per_mag if fewer images available
            while len(tensors) < self.n_per_mag:
                tensors.append(tensors[rng.integers(len(tensors))])
            imgs_per_mag.append(tensors)   # list of n_per_mag tensors
        return imgs_per_mag, int(info['binary']), int(info['subtype'])


def mil_collate_fn(batch):
    """
    Custom collate: stacks images_per_mag into [B, N, 3, H, W] tensors.
    Output per magnification: tensor of shape [B, N, 3, H, W]
    where N = IMAGES_PER_MAG.
    """
    imgs_list  = [item[0] for item in batch]   # list of B x (4 x N x tensor)
    y_bin      = torch.tensor([item[1] for item in batch], dtype=torch.long)
    y_sub      = torch.tensor([item[2] for item in batch], dtype=torch.long)
    # imgs_list[b][mag_idx][img_idx] = tensor [3,H,W]
    mag_batches = []
    for mag_idx in range(len(MAGNIFICATIONS)):
        # Stack: [B, N, 3, H, W]
        mag_tensor = torch.stack([
            torch.stack(imgs_list[b][mag_idx])   # [N, 3, H, W]
            for b in range(len(batch))
        ])  # [B, N, 3, H, W]
        mag_batches.append(mag_tensor)
    return (*mag_batches, y_bin, y_sub)


def make_loader(patients, augment, shuffle, img_size=IMG_SIZE, drop_last=None):
    ds  = MultiMagDataset(patients, patient_info, augment=augment,
                          img_size=img_size, n_per_mag=IMAGES_PER_MAG)
    dl  = drop_last if drop_last is not None else shuffle
    if shuffle:
        labels  = [patient_info[p]['binary'] for p in patients]
        counts  = [labels.count(0), labels.count(1)]
        weights = [1.0/counts[l] for l in labels]
        sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
        return DataLoader(ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=False,
                          drop_last=True, collate_fn=mil_collate_fn)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=False,
                      drop_last=dl, collate_fn=mil_collate_fn)


train_loader_warmup = make_loader(train_patients, augment=True,  shuffle=True,  img_size=IMG_SIZE_WARMUP)
train_loader        = make_loader(train_patients, augment=True,  shuffle=True,  img_size=IMG_SIZE)
val_loader          = make_loader(val_patients,   augment=False, shuffle=False, img_size=IMG_SIZE)
test_loader         = make_loader(test_patients,  augment=False, shuffle=False, img_size=IMG_SIZE)

print(f'Loaders: train={len(train_loader.dataset)} val={len(val_loader.dataset)} test={len(test_loader.dataset)}')
print(f'MIL: each patient → {IMAGES_PER_MAG} images/mag processed individually')

# Sanity check: verify batch shape is [B, N, 3, H, W]
print('Sanity check...')
_b = next(iter(train_loader))
print(f'  Per-mag shape (B,N,C,H,W): {_b[0].shape}')
print(f'  Binary labels: {_b[4].tolist()}')
del _b
print('DataLoader OK — MIL batch format confirmed.')

# ─────────────────────────────────────────────────────────────────────
# Class definitions moved here so they are available before Section 8
# (McNemar / baseline CV cells run before Section 10 training calls)
# ─────────────────────────────────────────────────────────────────────
from torchvision import transforms as T

class SingleMagDataset(Dataset):
    """
    Single-magnification baseline dataset — MIL version.
    Returns [N, 3, H, W] per sample so baseline also processes N images
    (ensuring fair comparison: same data richness as MAGNET).
    """
    def __init__(self, patient_ids, patient_info, mag, augment=False,
                 img_size=224, n_per_mag=IMAGES_PER_MAG):
        self.patient_ids  = patient_ids
        self.patient_info = patient_info
        self.mag          = mag
        self.img_size     = img_size
        self.n_per_mag    = n_per_mag
        self.pipeline     = albu_train if augment else albu_val

    def __len__(self): return len(self.patient_ids)

    def _load(self, path):
        img = Image.open(path).convert('RGB')
        img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        img = macenko_norm(img)
        return self.pipeline(image=np.array(img, dtype=np.uint8))['image']

    def __getitem__(self, idx):
        pid   = self.patient_ids[idx]
        info  = self.patient_info[pid]
        paths = info['paths'][self.mag]
        rng   = np.random.default_rng()
        n     = min(self.n_per_mag, len(paths)) if paths else 1
        chosen= rng.choice(len(paths), size=n, replace=False) if paths else [0]
        tensors = [self._load(paths[c]) for c in chosen] if paths else \
                  [torch.zeros(3, self.img_size, self.img_size)]
        while len(tensors) < self.n_per_mag:
            tensors.append(tensors[rng.integers(len(tensors))])
        # [N, 3, H, W]
        return torch.stack(tensors), int(info['binary'])


def train_single_baseline(backbone_name, mag, epochs=20, patience=6):
    """Single backbone baseline — MIL version for fair comparison."""
    print(f'\nBaseline: {backbone_name} @ {mag} (MIL, N={IMAGES_PER_MAG})')
    def _sl(patients, augment, sh):
        ds = SingleMagDataset(patients, patient_info, mag,
                              augment=augment, img_size=IMG_SIZE)
        return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=sh,
                          num_workers=NUM_WORKERS, pin_memory=False, drop_last=sh)
    tl=_sl(train_patients,True,True)
    vl=_sl(val_patients,False,False)
    el=_sl(test_patients,False,False)
    torch.cuda.empty_cache()
    # Backbone with MIL attention pooling for fair comparison
    backbone = timm.create_model(backbone_name, pretrained=True,
                                  num_classes=0, global_pool='avg').to(device)
    in_dim   = backbone(torch.randn(1,3,IMG_SIZE,IMG_SIZE).to(device)).shape[1]
    attn_pool= nn.Sequential(nn.Linear(in_dim,128), nn.Tanh(),
                              nn.Linear(128,1)).to(device)
    classifier = nn.Sequential(nn.Linear(in_dim,256), nn.BatchNorm1d(256),
                                nn.GELU(), nn.Dropout(0.4),
                                nn.Linear(256,2)).to(device)
    params = list(backbone.parameters()) + list(attn_pool.parameters()) + \
             list(classifier.parameters())
    opt = optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
    _s_opt_steps = math.ceil(len(tl)/ACCUM_STEPS)*epochs
    sch = optim.lr_scheduler.OneCycleLR(opt, max_lr=LR*5,
              total_steps=_s_opt_steps, pct_start=0.1)
    # CrossEntropyLoss here (FocalLoss defined later in notebook;
    # baseline uses CE for simplicity — fair since both use label smoothing)
    ce  = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    sc  = make_scaler()
    best_acc=0.0; pat=0
    ckpt=f'checkpoints/baseline_{_safe_name(backbone_name)}_{mag}.pth'
    for ep in range(1,epochs+1):
        backbone.train(); attn_pool.train(); classifier.train()
        opt.zero_grad()
        for step,(imgs,labels) in enumerate(tl):
            # imgs: [B, N, 3, H, W]
            imgs,labels = imgs.to(device), labels.to(device)
            B,N,C,H,W  = imgs.shape
            with autocast_ctx():
                feats = backbone(imgs.view(B*N,C,H,W)).view(B,N,-1)
                attn  = torch.softmax(attn_pool(feats), dim=1)
                pooled= (attn * feats).sum(1)
                loss  = ce(classifier(pooled), labels) / ACCUM_STEPS
            sc.scale(loss).backward()
            if (step+1)%ACCUM_STEPS==0 or (step+1)==len(tl):
                sc.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                sc.step(opt); sc.update(); opt.zero_grad(); sch.step()
        backbone.eval(); attn_pool.eval(); classifier.eval()
        preds,labs=[],[]
        with torch.no_grad():
            for imgs,labels in vl:
                B,N,C,H,W = imgs.shape
                imgs = imgs.to(device)
                feats = backbone(imgs.view(B*N,C,H,W)).view(B,N,-1)
                attn  = torch.softmax(attn_pool(feats),dim=1)
                pooled= (attn*feats).sum(1)
                preds.extend(classifier(pooled).argmax(1).cpu().tolist())
                labs.extend(labels.tolist())
        va=accuracy_score(labs,preds)
        if va>best_acc:
            best_acc=va
            torch.save({'bb':backbone.state_dict(),'attn':attn_pool.state_dict(),
                        'cls':classifier.state_dict()}, ckpt)
            pat=0
        else:
            pat+=1
            if pat>=patience: break
        print(f'  ep{ep:02d}: val_acc={va:.4f}')
    # Test evaluation
    ckpt_d = torch.load(ckpt, map_location=device)
    backbone.load_state_dict(ckpt_d['bb'])
    attn_pool.load_state_dict(ckpt_d['attn'])
    classifier.load_state_dict(ckpt_d['cls'])
    backbone.eval(); attn_pool.eval(); classifier.eval()
    preds,labs,probs=[],[],[]
    with torch.no_grad():
        for imgs,labels in el:
            B,N,C,H,W = imgs.shape
            imgs = imgs.to(device)
            feats = backbone(imgs.view(B*N,C,H,W)).view(B,N,-1)
            attn  = torch.softmax(attn_pool(feats),dim=1)
            logits= classifier((attn*feats).sum(1))
            preds.extend(logits.argmax(1).cpu().tolist())
            probs.extend(F.softmax(logits,1)[:,1].cpu().tolist())
            labs.extend(labels.tolist())
    ta=accuracy_score(labs,preds)
    f1b=f1_score(labs,preds,average='weighted',zero_division=0)
    try: ta_auc=roc_auc_score(labs,probs)
    except ValueError: ta_auc=float('nan')
    del backbone,attn_pool,classifier; torch.cuda.empty_cache()
    print(f'  → acc={ta:.4f}  auc={ta_auc:.4f}  f1={f1b:.4f}')
    return {'config':f'{backbone_name}@{mag}','test_acc':ta,'auc':ta_auc,'f1':f1b}

# ── ConcatFusion Baseline (MIL-compatible) ───────────────────────────────────
class ConcatFusionBaseline(nn.Module):
    """
    Simple multi-magnification baseline: concatenate projected features from
    all 4 magnifications.
    MIL-compatible: applies attention pooling over N images before projection,
    matching the same data richness as MAGNET for a fair comparison.
    """
    def __init__(self, proj_dim=256, num_binary=2, pretrained=True):
        super().__init__()
        self.branches = nn.ModuleDict({
            mag: timm.create_model(BACKBONES[mag], pretrained=pretrained,
                                   num_classes=0, global_pool='avg')
            for mag in MAGNIFICATIONS
        })
        # Attention MIL pooling per branch (same as BranchExtractor)
        self.attn_pools = nn.ModuleDict({
            mag: nn.Sequential(
                nn.Linear(BACKBONE_OUT_DIM[mag], 128),
                nn.Tanh(),
                nn.Linear(128, 1)
            ) for mag in MAGNIFICATIONS
        })
        self.projs = nn.ModuleDict({
            mag: nn.Linear(BACKBONE_OUT_DIM[mag], proj_dim)
            for mag in MAGNIFICATIONS
        })
        self.classifier = nn.Sequential(
            nn.Linear(4 * proj_dim, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_binary)
        )

    def forward(self, x40, x100, x200, x400):
        inputs = {'40X':x40, '100X':x100, '200X':x200, '400X':x400}
        feats  = []
        for mag in MAGNIFICATIONS:
            x = inputs[mag]                              # [B, N, 3, H, W]
            B, N, C, H, W = x.shape
            f    = self.branches[mag](x.view(B*N, C, H, W))   # [B*N, D]
            f    = f.view(B, N, -1)                           # [B, N, D]
            attn = torch.softmax(self.attn_pools[mag](f), dim=1)  # [B, N, 1]
            pooled = (attn * f).sum(dim=1)               # [B, D]
            feats.append(self.projs[mag](pooled))         # [B, proj_dim]
        return self.classifier(torch.cat(feats, dim=1))  # [B, 2]


def run_concat_baseline(epochs=20, patience=6):
    print('\nBaseline: ConcatFusion (4-mag MIL attention pooling, no cross-attn)')
    torch.cuda.empty_cache()
    m   = ConcatFusionBaseline(pretrained=True).to(device)
    opt = optim.AdamW(m.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    _cc_opt_steps = math.ceil(len(train_loader)/ACCUM_STEPS)*epochs
    sch = optim.lr_scheduler.OneCycleLR(
        opt, max_lr=LR*5,
        total_steps=_cc_opt_steps,
        pct_start=0.1, anneal_strategy='cos',
        div_factor=10, final_div_factor=1e3)
    ce  = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    sc  = make_scaler()
    best_acc=0.0; pat=0; ckpt='checkpoints/concat_baseline.pth'

    for ep in range(1, epochs+1):
        m.train(); opt.zero_grad()
        for step, batch in enumerate(train_loader):
            x40,x100,x200,x400,y_bin,_ = [b.to(device) for b in batch]
            with autocast_ctx():
                loss = ce(m(x40,x100,x200,x400), y_bin) / ACCUM_STEPS
            sc.scale(loss).backward()
            if (step+1) % ACCUM_STEPS == 0 or (step+1) == len(train_loader):
                sc.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                sc.step(opt); sc.update(); opt.zero_grad(); sch.step()
        m.eval(); preds, labs = [], []
        with torch.no_grad():
            for batch in val_loader:
                x40,x100,x200,x400,y_bin,_ = [b.to(device) for b in batch]
                preds.extend(m(x40,x100,x200,x400).argmax(1).cpu().tolist())
                labs.extend(y_bin.cpu().tolist())
        va = accuracy_score(labs, preds)
        if va > best_acc:
            best_acc=va; torch.save(m.state_dict(), ckpt); pat=0
        else:
            pat+=1
            if pat >= patience: break
        print(f'  ep{ep:02d}: val_acc={va:.4f}')

    m.load_state_dict(torch.load(ckpt, map_location=device)); m.eval()
    preds, labs, probs = [], [], []
    with torch.no_grad():
        for batch in test_loader:
            x40,x100,x200,x400,y_bin,_ = [b.to(device) for b in batch]
            logits = m(x40,x100,x200,x400)
            preds.extend(logits.argmax(1).cpu().tolist())
            probs.extend(F.softmax(logits,1)[:,1].cpu().tolist())
            labs.extend(y_bin.cpu().tolist())
    ta   = accuracy_score(labs, preds)
    f1c  = f1_score(labs, preds, average='weighted', zero_division=0)
    try: ta_auc = roc_auc_score(labs, probs)
    except ValueError: ta_auc = float('nan')
    del m; torch.cuda.empty_cache()
    print(f'  → acc={ta:.4f}  auc={ta_auc:.4f}  f1={f1c:.4f}')
    return {'config':'ConcatFusion (4-mag MIL)', 'test_acc':ta, 'auc':ta_auc, 'f1':f1c}


Loaders: train=57 val=12 test=13
MIL: each patient → 3 images/mag processed individually
Sanity check...
  Per-mag shape (B,N,C,H,W): torch.Size([4, 3, 3, 224, 224])
  Binary labels: [1, 0, 1, 1]
DataLoader OK — MIL batch format confirmed.


## 4. Model: MAGNET

### 4.1 Branch Extractor

In [7]:
class BranchExtractor(nn.Module):
    """
    Pretrained backbone + MIL pooling.
    Input:  x of shape [B, N, 3, H, W]  (N images per patient per mag)
    Process: run each of the N images through backbone independently,
             then aggregate via attention pooling over the N feature vectors.
    Output: raw [B, backbone_dim], proj [B, feat_dim]
    """
    def __init__(self, backbone_name, in_dim, out_dim, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained,
                                          num_classes=0, global_pool='avg')
        # Attention MIL pooling — learns which images are most informative
        self.attn_pool = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1)   # scalar attention score per image
        )
        self.proj = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.LayerNorm(out_dim),
            nn.GELU(),
        )

    def forward(self, x):
        """
        x: [B, N, 3, H, W]
        Returns: raw [B, in_dim], proj [B, out_dim]
        """
        B, N, C, H, W = x.shape
        # Flatten batch+instance dims → process all N*B images in one pass
        x_flat  = x.view(B * N, C, H, W)            # [B*N, 3, H, W]
        feats   = self.backbone(x_flat)              # [B*N, in_dim]
        feats   = feats.view(B, N, -1)               # [B, N, in_dim]
        # Attention MIL pooling: learn a weight for each of the N images
        attn    = self.attn_pool(feats)              # [B, N, 1]
        attn    = torch.softmax(attn, dim=1)         # normalise over N
        raw     = (attn * feats).sum(dim=1)          # [B, in_dim] weighted sum
        proj    = self.proj(raw)                     # [B, out_dim]
        return raw, proj

print('BranchExtractor (MIL attention pooling over N images) defined.')


BranchExtractor (MIL attention pooling over N images) defined.


### 4.2 Cross-Magnification Attention Fusion

In [8]:
class CrossMagFusion(nn.Module):
    """Multi-head self-attention across 4 magnification tokens."""
    def __init__(self, feat_dim=512, num_heads=8, dropout=0.1):
        super().__init__()
        assert feat_dim % num_heads == 0
        self.attn      = nn.MultiheadAttention(feat_dim, num_heads, dropout=dropout, batch_first=True)
        self.mag_scale = nn.Parameter(torch.ones(4))
        self.ffn  = nn.Sequential(
            nn.Linear(feat_dim, feat_dim*2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(feat_dim*2, feat_dim))
        self.norm1 = nn.LayerNorm(feat_dim)
        self.norm2 = nn.LayerNorm(feat_dim)

    def forward(self, feats):
        x = torch.stack(feats, dim=1)                        # [B,4,D]
        attn_out, attn_w = self.attn(x, x, x)
        x = self.norm1(x + attn_out)
        w = F.softmax(self.mag_scale, dim=0)
        fused = (x * w.unsqueeze(0).unsqueeze(-1)).sum(1)    # [B,D]
        fused = self.norm2(fused + self.ffn(fused))
        return fused, attn_w

print('CrossMagFusion defined.')

CrossMagFusion defined.


### 4.3 Batch Graph Network (pure PyTorch)

In [9]:
class SparseGraphConv(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
    def forward(self, h, adj):
        return F.relu(adj @ self.W(h))

class BatchGraphNet(nn.Module):
    """Treats each batch as a graph; edges = cosine similarity > threshold."""
    def __init__(self, feat_dim=512, hidden_dim=256, num_layers=2, threshold=0.5, dropout=0.1):
        super().__init__()
        self.threshold = threshold
        in_d = feat_dim
        layers = []
        for _ in range(num_layers):
            layers.append(SparseGraphConv(in_d, hidden_dim)); in_d = hidden_dim
        self.gcn_layers = nn.ModuleList(layers)
        self.drop    = nn.Dropout(dropout)
        self.readout = nn.Linear(hidden_dim, feat_dim)
        self.norm    = nn.LayerNorm(feat_dim)

    def _adj(self, h):
        norm = F.normalize(h.detach(), p=2, dim=1)
        sim  = torch.mm(norm, norm.t())
        mask = (sim > self.threshold).float()
        mask.fill_diagonal_(1.0)
        deg = mask.sum(1, keepdim=True).clamp(min=1)
        d   = deg.pow(-0.5)
        return mask * d * d.t()

    def forward(self, h):
        adj = self._adj(h)
        out = h
        for layer in self.gcn_layers:
            out = self.drop(layer(out, adj))
        return self.norm(h + self.readout(out))

print('BatchGraphNet defined (pure PyTorch, no DGL).')

BatchGraphNet defined (pure PyTorch, no DGL).


### 4.4 Full MAGNET Model

In [10]:
class TransformerMagFusion(nn.Module):
    """
    Phase 2.2: Full Transformer encoder over 4 magnification tokens.
    Adds a learnable [CLS] token whose output is the fused representation.
    Uses 4 encoder layers with 8 heads — deeper than original single-attn.
    """
    def __init__(self, feat_dim=512, num_heads=8, num_layers=4, dropout=0.1):
        super().__init__()
        self.cls_token = nn.Parameter(torch.zeros(1, 1, feat_dim))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.pos_embed = nn.Parameter(torch.zeros(1, 5, feat_dim))  # 4 mags + CLS
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=feat_dim, nhead=num_heads,
            dim_feedforward=feat_dim*4, dropout=dropout,
            batch_first=True, norm_first=True   # pre-norm for stability
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm    = nn.LayerNorm(feat_dim)

    def forward(self, feats):
        """
        feats: list of 4 tensors [B, feat_dim]
        returns: cls_out [B, feat_dim], attn_weights [B, 4, 4] (from last layer)
        """
        B = feats[0].size(0)
        x = torch.stack(feats, dim=1)                        # [B, 4, D]
        cls = self.cls_token.expand(B, -1, -1)              # [B, 1, D]
        x   = torch.cat([cls, x], dim=1)                    # [B, 5, D]
        x   = x + self.pos_embed
        x   = self.encoder(x.float()).to(dtype=x.dtype)       # fp32 for stability
        cls_out = self.norm(x[:, 0])                        # [B, D] — CLS token
        # Return token-to-token attention from mag tokens only (for viz)
        attn_w = torch.zeros(B, 4, 4, device=x.device)     # placeholder
        return cls_out, attn_w


class MAGNET(nn.Module):
    """
    MAGNET v2 — upgraded for performance:
    - Stronger backbones (EfficientNet-B2/B1, ResNet50, EfficientNet-B0)
    - Transformer encoder fusion with CLS token (4 layers, 8 heads)
    - Learnable attention GNN
    - 3-head ensemble binary classifier (averaged at inference)
    - Focal Loss compatible outputs
    """
    def __init__(self, feat_dim=FEAT_DIM, num_binary=2, num_subtype=8,
                 pretrained=True, use_gnn=True, use_aux=True):
        super().__init__()
        self.use_gnn = use_gnn
        self.use_aux = use_aux

        # ── Backbone branches ───────────────────────────────────────────────
        self.branches = nn.ModuleDict({
            mag: BranchExtractor(BACKBONES[mag], BACKBONE_OUT_DIM[mag],
                                 feat_dim, pretrained)
            for mag in MAGNIFICATIONS
        })

        # ── Transformer fusion ──────────────────────────────────────────────
        self.fusion = TransformerMagFusion(feat_dim, num_heads=8, num_layers=4)

        # ── GNN ────────────────────────────────────────────────────────────
        if use_gnn:
            self.gnn = BatchGraphNet(feat_dim, hidden_dim=256)

        # ── Multi-head ensemble binary classifier (Phase 2.4) ───────────────
        def _bin_head():
            return nn.Sequential(
                nn.Linear(feat_dim, 256), nn.BatchNorm1d(256), nn.GELU(),
                nn.Dropout(0.4), nn.Linear(256, num_binary))

        self.binary_heads = nn.ModuleList([_bin_head() for _ in range(3)])

        # ── Subtype head (auxiliary regulariser) ────────────────────────────
        self.subtype_head = nn.Sequential(
            nn.Linear(feat_dim, 256), nn.BatchNorm1d(256), nn.GELU(),
            nn.Dropout(0.4), nn.Linear(256, num_subtype))

        # ── Uncertainty head ────────────────────────────────────────────────
        self.uncertainty_head = nn.Sequential(
            nn.Linear(feat_dim, 128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, 1), nn.Sigmoid())

        # ── Auxiliary per-branch heads ──────────────────────────────────────
        if use_aux:
            self.aux_heads = nn.ModuleDict({
                mag: nn.Linear(BACKBONE_OUT_DIM[mag], num_binary)
                for mag in MAGNIFICATIONS
            })

    @property
    def binary_head(self):
        """Compatibility shim: returns the first head (used by GradCAM etc.)"""
        return self.binary_heads[0]

    def _ensemble_binary(self, fused):
        """Average logits across the 3 binary heads."""
        return torch.stack([h(fused) for h in self.binary_heads], dim=0).mean(0)

    def forward(self, x40, x100, x200, x400, return_attn=False):
        inputs = {'40X':x40,'100X':x100,'200X':x200,'400X':x400}
        raws, projs, aux_logits = {}, {}, {}
        for mag in MAGNIFICATIONS:
            raw, proj = self.branches[mag](inputs[mag])
            raws[mag]=raw; projs[mag]=proj
            if self.use_aux:
                aux_logits[mag] = self.aux_heads[mag](raw)
        fused, attn_w = self.fusion([projs[m] for m in MAGNIFICATIONS])
        if self.use_gnn:
            fused = self.gnn(fused)
        bl  = self._ensemble_binary(fused)
        sl  = self.subtype_head(fused)
        unc = self.uncertainty_head(fused)
        out = (bl, sl, unc, list(aux_logits.values()) if self.use_aux else [])
        return (*out, attn_w) if return_attn else out

    @torch.no_grad()
    def mc_uncertainty(self, x40, x100, x200, x400, n_passes=20):
        self.train()
        for m in self.modules():
            if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
                m.eval()
        probs = []
        for _ in range(n_passes):
            logits, *_ = self(x40, x100, x200, x400)
            probs.append(F.softmax(logits, dim=1).unsqueeze(0))
        self.eval()
        probs = torch.cat(probs, dim=0)
        return probs.mean(0), probs.var(0)

print('MAGNET v2 defined (TransformerFusion + 3-head ensemble).')


MAGNET v2 defined (TransformerFusion + 3-head ensemble).


### 4.5 Smoke Test

In [11]:
torch.cuda.empty_cache()
_m = MAGNET(pretrained=False).to(device)
total_p = sum(p.numel() for p in _m.parameters())
bb_p    = sum(p.numel() for mg in MAGNIFICATIONS
               for p in _m.branches[mg].backbone.parameters())
print(f'Total params             : {total_p/1e6:.2f} M')
print(f'Backbone params (low LR) : {bb_p/1e6:.2f} M')
print(f'Head + fusion params     : {(total_p-bb_p)/1e6:.2f} M')

# MIL input: [B, N, 3, H, W]
_d = [torch.randn(2, IMAGES_PER_MAG, 3, IMG_SIZE, IMG_SIZE).to(device)
      for _ in range(4)]
with autocast_ctx():
    bl,sl,unc,aux,aw = _m(*_d, return_attn=True)
print(f'binary:{bl.shape} subtype:{sl.shape} unc:{unc.shape} '
      f'aux:{[a.shape for a in aux]} attn:{aw.shape}')
if device.type=='cuda':
    used = torch.cuda.memory_allocated()/1024**2
    print(f'VRAM (B=2, N={IMAGES_PER_MAG}): {used:.0f} MB')
del _m, _d; torch.cuda.empty_cache()

# MC-Dropout live check
_m2 = MAGNET(pretrained=False).to(device)
_x  = [torch.randn(2, IMAGES_PER_MAG, 3, IMG_SIZE, IMG_SIZE).to(device)
       for _ in range(4)]
with torch.no_grad():
    r1, _ = _m2.mc_uncertainty(*_x, n_passes=2)
    r2, _ = _m2.mc_uncertainty(*_x, n_passes=2)
assert not torch.allclose(r1, r2, atol=1e-6), 'MC-Dropout not active!'
del _m2, _x; torch.cuda.empty_cache()
print('MC-Dropout verified active.')
print('Smoke test passed.')


Total params             : 59.14 M
Backbone params (low LR) : 41.73 M
Head + fusion params     : 17.41 M
binary:torch.Size([2, 2]) subtype:torch.Size([2, 8]) unc:torch.Size([2, 1]) aux:[torch.Size([2, 2]), torch.Size([2, 2]), torch.Size([2, 2]), torch.Size([2, 2])] attn:torch.Size([2, 4, 4])
VRAM (B=2, N=3): 1615 MB
MC-Dropout verified active.
Smoke test passed.


## 5. Training Pipeline

### 5.1 Loss Function (with Label Smoothing)

In [12]:
class FocalLoss(nn.Module):
    """
    Focal Loss computed in float32 — prevents fp16 NaN under AMP.
    fp16: exp(-ce) can underflow → 0 → (1-0)^gamma * inf = nan.
    Clamping ce before exp and forcing float32 eliminates this.
    """
    def __init__(self, gamma=FOCAL_GAMMA, weight=None, label_smoothing=0.0):
        super().__init__()
        self.gamma           = gamma
        self.weight          = weight
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        logits = logits.float()   # always fp32 — prevents AMP NaN
        if self.weight is not None:
            self.weight = self.weight.to(logits.device).float()
        ce  = F.cross_entropy(logits, targets, weight=self.weight,
                              label_smoothing=self.label_smoothing, reduction='none')
        pt  = torch.exp(-ce.clamp(max=80))  # clamp prevents exp overflow
        fl  = (1.0 - pt).pow(self.gamma) * ce
        return fl.mean()


class MAGNETLoss(nn.Module):
    """L = α·FL_binary + β·CE_subtype + γ·CE_aux + δ·L_unc.
    All losses cast to fp32 before computation."""
    def __init__(self, alpha=1.0, beta=0.2, gamma_aux=0.3, delta=0.01,
                 class_weights=None, label_smoothing=LABEL_SMOOTHING):
        super().__init__()
        self.alpha=alpha; self.beta=beta; self.gamma_aux=gamma_aux; self.delta=delta
        w = class_weights if class_weights is not None else torch.tensor([0.69,0.31])
        self.focal_binary = FocalLoss(gamma=FOCAL_GAMMA, weight=w,
                                       label_smoothing=label_smoothing)
        self.ce_subtype   = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
        self.ce_aux       = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
        self.mse          = nn.MSELoss()

    def forward(self, bl, sl, unc, aux, y_bin, y_sub):
        L_bin = self.focal_binary(bl, y_bin)
        L_sub = self.ce_subtype(sl.float(), y_sub)
        L_aux = torch.tensor(0.0, device=bl.device)
        if aux:
            for a in aux: L_aux = L_aux + self.ce_aux(a.float(), y_bin)
            L_aux = L_aux / len(aux)
        with torch.no_grad():
            conf  = F.softmax(bl.float(), dim=1).max(1).values
            t_unc = (1.0 - conf).unsqueeze(1)
        L_unc = self.mse(unc.float(), t_unc)
        total = self.alpha*L_bin + self.beta*L_sub + self.gamma_aux*L_aux + self.delta*L_unc
        return total, {'binary':L_bin.item(),'subtype':L_sub.item(),
                       'aux':L_aux.item(),'unc':L_unc.item()}

print('FocalLoss (fp32-safe) + MAGNETLoss defined.')


FocalLoss (fp32-safe) + MAGNETLoss defined.


### 5.2 Train / Eval Loops

In [13]:
def make_scaler():
    try: return torch.amp.GradScaler(device='cuda')
    except TypeError: return torch.cuda.amp.GradScaler()


def train_epoch(model, loader, optimizer, criterion, scaler, device,
                 accum_steps=ACCUM_STEPS, scheduler=None):
    """
    Training loop with gradient accumulation.
    Batch format: (x40[B,N,C,H,W], x100, x200, x400, y_bin, y_sub)
    """
    model.train()
    tot_loss=0.0; all_preds=[]; all_labels=[]
    optimizer.zero_grad()
    for step, batch in enumerate(tqdm(loader, desc='Train', leave=False)):
        x40,x100,x200,x400,y_bin,y_sub = [b.to(device) for b in batch]
        with autocast_ctx():
            bl,sl,unc,aux = model(x40,x100,x200,x400)
            loss,_ = criterion(bl,sl,unc,aux,y_bin,y_sub)
            loss   = loss / accum_steps
        scaler.scale(loss).backward()
        if (step+1) % accum_steps == 0 or (step+1) == len(loader):
            scaler.unscale_(optimizer)
            total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            if torch.isfinite(total_norm):
                scaler.step(optimizer)
                if scheduler is not None:
                    scheduler.step()
            else:
                print(f'  ⚠ NaN/Inf grad norm={total_norm:.2f}, skipping step')
            scaler.update()
            optimizer.zero_grad()
        tot_loss  += loss.item() * accum_steps
        all_preds.extend(bl.argmax(1).cpu().tolist())
        all_labels.extend(y_bin.cpu().tolist())
    return tot_loss / len(loader), accuracy_score(all_labels, all_preds)


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    """Standard evaluation (no TTA). Batch format: [B,N,C,H,W] per mag."""
    model.eval()
    tot_loss=0.0
    all_preds=[]; all_labels=[]
    all_sub_preds=[]; all_subs=[]
    all_unc=[]; all_probs=[]
    for batch in tqdm(loader, desc='Eval', leave=False):
        x40,x100,x200,x400,y_bin,y_sub = [b.to(device) for b in batch]
        with autocast_ctx():
            bl,sl,unc,aux = model(x40,x100,x200,x400)
            loss,_        = criterion(bl,sl,unc,aux,y_bin,y_sub)
        tot_loss += loss.item()
        all_probs.extend(F.softmax(bl.float(),1)[:,1].cpu().tolist())
        all_preds.extend(bl.argmax(1).cpu().tolist())
        all_labels.extend(y_bin.cpu().tolist())
        all_sub_preds.extend(sl.argmax(1).cpu().tolist())
        all_subs.extend(y_sub.cpu().tolist())
        all_unc.extend(unc.squeeze(1).float().cpu().tolist())
    acc     = accuracy_score(all_labels, all_preds)
    sub_acc = accuracy_score(all_subs, all_sub_preds)
    try: auc = roc_auc_score(all_labels, all_probs)
    except ValueError: auc = float('nan')
    return (tot_loss/len(loader), acc, sub_acc, auc,
            all_preds, all_labels, all_sub_preds, all_subs, all_unc, all_probs)


def tta_eval(model, loader, device, tta_transforms=TTA_TRANSFORMS):
    """
    Proper Test-Time Augmentation:
    For each test batch, apply each TTA transform to the SAME images
    (reprocessed from the already-loaded numpy arrays stored in the dataset),
    run through model, average predicted probabilities across transforms.

    Strategy: rebuild the loader with each TTA transform's pipeline,
    keeping the SAME random seed so the same images are selected,
    but different geometric transforms applied.
    """
    model.eval()
    patients = loader.dataset.patient_ids
    true_labs = [patient_info[p]['binary'] for p in patients]
    all_run_probs = []

    for tf_idx, tta_tf in enumerate(tta_transforms):
        # Build a dataset that uses this specific TTA transform
        class TTADataset(MultiMagDataset):
            def __init__(self, patient_ids, patient_info, img_size):
                super().__init__(patient_ids, patient_info,
                                 augment=False, img_size=img_size,
                                 n_per_mag=1)  # 1 image per mag for TTA speed
                self.pipeline = tta_tf  # override pipeline

        tta_ds = TTADataset(patients, patient_info, IMG_SIZE)
        tta_loader = DataLoader(tta_ds, batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=NUM_WORKERS, pin_memory=False,
                                collate_fn=mil_collate_fn)
        run_probs = []
        with torch.no_grad():
            for batch in tta_loader:
                x40,x100,x200,x400,_,_ = [b.to(device) for b in batch]
                with autocast_ctx():
                    bl,*_ = model(x40,x100,x200,x400)
                run_probs.extend(F.softmax(bl.float(),1)[:,1].cpu().tolist())
        all_run_probs.append(run_probs)
        print(f'  TTA pass {tf_idx+1}/{len(tta_transforms)} done')

    avg_probs = np.array(all_run_probs).mean(axis=0)
    preds     = (avg_probs >= 0.5).astype(int).tolist()
    return preds, true_labs, avg_probs.tolist()


print('Training loops + proper TTA defined.')


Training loops + proper TTA defined.


### 5.3 Main Training Run

In [15]:
import math
torch.cuda.empty_cache()
model = MAGNET(feat_dim=FEAT_DIM, pretrained=True, use_gnn=True, use_aux=True).to(device)

bb_params  = [p for mg in MAGNIFICATIONS for p in model.branches[mg].backbone.parameters()]
oth_params = [p for n,p in model.named_parameters()
              if not any(n.startswith(f'branches.{mg}.backbone') for mg in MAGNIFICATIONS)]

optimizer = optim.AdamW([
    {'params': bb_params,  'lr': LR_BACKBONE},
    {'params': oth_params, 'lr': LR},
], weight_decay=WEIGHT_DECAY)

batches_per_ep   = len(train_loader)
opt_steps_per_ep = math.ceil(batches_per_ep / ACCUM_STEPS)
total_opt_steps  = EPOCHS * opt_steps_per_ep
print(f'Batches/epoch={batches_per_ep}  Opt-steps/epoch={opt_steps_per_ep}  Total opt-steps={total_opt_steps}')

scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=[LR_BACKBONE * 5, LR * 5],
    total_steps=total_opt_steps,
    pct_start=0.1, anneal_strategy='cos',
    div_factor=10, final_div_factor=1e3,
)

criterion = MAGNETLoss()
scaler    = make_scaler()

# Safe VRAM check: eval mode (avoids BN batch-size-1 error) + B=1,N=1
if device.type == 'cuda':
    torch.cuda.reset_peak_memory_stats()
    model.eval()                          # ← KEY FIX: eval mode for BN
    with torch.no_grad(), autocast_ctx():
        _d = [torch.randn(1, 1, 3, IMG_SIZE, IMG_SIZE, device=device) for _ in range(4)]
        _ = model(*_d)
    del _d, _
    model.train()                         # ← restore training mode
    torch.cuda.empty_cache()
    peak_1 = torch.cuda.max_memory_allocated() / 1024**2
    est    = peak_1 * BATCH_SIZE * IMAGES_PER_MAG
    print(f'Est. peak VRAM (B={BATCH_SIZE},N={IMAGES_PER_MAG}): ~{est:.0f} MB / 4096 MB')
    if est > 3500:
        print('⚠ Estimated VRAM tight — if OOM: set IMAGES_PER_MAG=2 or BATCH_SIZE=2 in config')
    else:
        print('✓ VRAM estimate looks safe.')

best_val_acc=0.0; patience_cnt=0
history={'train_loss':[],'val_loss':[],'train_acc':[],'val_acc':[],'val_auc':[]}

print(f'Training up to {EPOCHS} epochs | MIL({IMAGES_PER_MAG}×/mag) | OneCycleLR | accum={ACCUM_STEPS}')
print(f'Progressive resize: {WARMUP_EPOCHS} epochs @ {IMG_SIZE_WARMUP}px → {IMG_SIZE}px')
print('='*70)

for epoch in range(1, EPOCHS+1):
    t0 = time.time()
    active_loader = train_loader_warmup if epoch <= WARMUP_EPOCHS else train_loader
    img_sz        = IMG_SIZE_WARMUP    if epoch <= WARMUP_EPOCHS else IMG_SIZE

    train_loss, train_acc = train_epoch(
        model, active_loader, optimizer, criterion, scaler, device,
        accum_steps=ACCUM_STEPS, scheduler=scheduler)
    val_loss, val_acc, val_sub_acc, val_auc, *_ = eval_epoch(
        model, val_loader, criterion, device)

    if not (math.isfinite(train_loss) and math.isfinite(val_loss)):
        print(f'⚠ NaN/Inf loss at epoch {epoch} — stopping. Check FocalLoss & LR.')
        break

    for k,v in zip(['train_loss','val_loss','train_acc','val_acc','val_auc'],
                   [train_loss,val_loss,train_acc,val_acc,val_auc]):
        history[k].append(v)

    vram = torch.cuda.memory_allocated()/1024**2 if device.type=='cuda' else 0
    print(f'Ep{epoch:3d}/{EPOCHS} [{img_sz}px] '
          f'TL={train_loss:.4f} TA={train_acc:.4f} | '
          f'VL={val_loss:.4f} VA={val_acc:.4f} SubA={val_sub_acc:.4f} AUC={val_auc:.4f} | '
          f'{time.time()-t0:.1f}s VRAM={vram:.0f}MB')

    if val_acc > best_val_acc:
        best_val_acc=val_acc
        torch.save(model.state_dict(),'checkpoints/best_model.pth')
        patience_cnt=0
        print(f'  ✓ Best val acc: {best_val_acc:.4f}')
    else:
        patience_cnt+=1
        if patience_cnt>=EARLY_STOP:
            print(f'Early stopping at epoch {epoch}'); break

print(f'\nBest val acc: {best_val_acc:.4f}')

Batches/epoch=14  Opt-steps/epoch=4  Total opt-steps=240
Est. peak VRAM (B=4,N=3): ~26177 MB / 4096 MB
⚠ Estimated VRAM tight — if OOM: set IMAGES_PER_MAG=2 or BATCH_SIZE=2 in config
Training up to 60 epochs | MIL(3×/mag) | OneCycleLR | accum=4
Progressive resize: 5 epochs @ 160px → 224px


Train:  29%|██▊       | 4/14 [00:17<00:34,  3.43s/it]

  ⚠ NaN/Inf grad norm=nan, skipping step


Train:  57%|█████▋    | 8/14 [00:26<00:14,  2.42s/it]

  ⚠ NaN/Inf grad norm=nan, skipping step


Train:  86%|████████▌ | 12/14 [00:34<00:04,  2.23s/it]

  ⚠ NaN/Inf grad norm=nan, skipping step


  ⚠ NaN/Inf grad norm=nan, skipping step


⚠ NaN/Inf loss at epoch 1 — stopping. Check FocalLoss & LR.

Best val acc: 0.0000


### 5.4 Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))
axes[0].plot(history['train_loss'],label='Train'); axes[0].plot(history['val_loss'],label='Val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(history['train_acc'],label='Train'); axes[1].plot(history['val_acc'],label='Val')
axes[1].set_title('Binary Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()
axes[2].plot(history['val_auc'],color='purple',label='Val AUC')
axes[2].set_title('AUC-ROC (Val)'); axes[2].set_xlabel('Epoch'); axes[2].legend()
plt.tight_layout()
plt.savefig('plots/training_curves.png',dpi=150); plt.show()

## 6. Test-Set Evaluation (Fixed Split)

In [ ]:
model.load_state_dict(torch.load('checkpoints/best_model.pth', map_location=device))
(test_loss, test_acc, test_sub_acc, test_auc,
 test_preds, test_labels, test_sub_preds, test_sub_labels,
 test_unc, test_probs) = eval_epoch(model, test_loader, criterion, device)

precision = precision_score(test_labels, test_preds, average='weighted', zero_division=0)
recall    = recall_score(test_labels, test_preds, average='weighted', zero_division=0)
f1        = f1_score(test_labels, test_preds, average='weighted', zero_division=0)
cm        = confusion_matrix(test_labels, test_preds)

print('='*60)
print(f'NOTE: Fixed-split test = {len(test_labels)} patients. '
      'Use 5-fold CV results (Section 7) for publication.')
print('='*60)
print(f'Test Accuracy (binary) : {test_acc:.4f}')
print(f'Test AUC-ROC           : {test_auc:.4f}')
print(f'Test F1 (weighted)     : {f1:.4f}')
print(f'Test Precision         : {precision:.4f}')
print(f'Test Recall            : {recall:.4f}')
print(f'Subtype Acc (auxiliary): {test_sub_acc:.4f}  ← regulariser, not primary metric')
print('='*60)
print(classification_report(test_labels, test_preds, target_names=['Benign','Malignant']))

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(13,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Benign','Malignant'], yticklabels=['Benign','Malignant'])
axes[0].set_title('Confusion Matrix — Binary'); axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
fpr,tpr,_ = roc_curve(test_labels, test_probs)
axes[1].plot(fpr,tpr,lw=2,label=f'AUC={test_auc:.4f}'); axes[1].plot([0,1],[0,1],'k--')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR'); axes[1].set_title('ROC Curve'); axes[1].legend()
plt.tight_layout(); plt.savefig('plots/evaluation_fixed_split.png',dpi=150); plt.show()

## 7. 5-Fold Patient-wise Cross-Validation

**Primary evaluation for the paper.**  
Uses all 82 patients with stratified splitting (preserves B/M ratio per fold).  
Reports 95% CI via t-distribution (df = N_FOLDS − 1 = 4).

In [ ]:
import math

def ci95(vals):
    n = len(vals); m = np.mean(vals)
    se = np.std(vals, ddof=1) / np.sqrt(n)
    h  = st.t.ppf(0.975, df=n-1) * se
    return m, h


skf          = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fold_results = []
fold_models  = []

print(f'{N_FOLDS}-Fold Patient-wise CV  (MIL + TTA + OneCycleLR)')
print('='*65)

for fold, (train_idx, test_idx) in enumerate(
        skf.split(all_patients, pat_labels)):

    fold_train_all = [all_patients[i] for i in train_idx]
    fold_test      = [all_patients[i] for i in test_idx]
    n_val          = max(2, int(0.15 * len(fold_train_all)))
    fold_val       = fold_train_all[-n_val:]
    fold_train     = fold_train_all[:-n_val]
    print(f'\nFold {fold+1}/{N_FOLDS} — train={len(fold_train)} val={len(fold_val)} test={len(fold_test)}')

    fl_train = make_loader(fold_train, augment=True,  shuffle=True,  img_size=IMG_SIZE)
    fl_val   = make_loader(fold_val,   augment=False, shuffle=False, img_size=IMG_SIZE)
    fl_test  = make_loader(fold_test,  augment=False, shuffle=False, img_size=IMG_SIZE)

    torch.cuda.empty_cache()
    m = MAGNET(feat_dim=FEAT_DIM, pretrained=True, use_gnn=True, use_aux=True).to(device)

    bb_p = [p for mg in MAGNIFICATIONS for p in m.branches[mg].backbone.parameters()]
    oth  = [p for n,p in m.named_parameters()
            if not any(n.startswith(f'branches.{mg}.backbone') for mg in MAGNIFICATIONS)]
    opt  = optim.AdamW([{'params':bb_p,'lr':LR_BACKBONE},
                        {'params':oth, 'lr':LR}], weight_decay=WEIGHT_DECAY)
    _opt_steps = math.ceil(len(fl_train) / ACCUM_STEPS) * KF_EPOCHS
    sch  = optim.lr_scheduler.OneCycleLR(
        opt, max_lr=[LR_BACKBONE*5, LR*5],
        total_steps=_opt_steps,
        pct_start=0.1, anneal_strategy='cos',
        div_factor=10, final_div_factor=1e3)
    crit = MAGNETLoss()
    sc   = make_scaler()

    best_acc=0.0; pat=0
    ckpt = os.path.join('checkpoints', f'fold_{fold+1}.pth')

    for ep in range(1, KF_EPOCHS+1):
        # train_epoch handles scheduler stepping internally
        train_epoch(m, fl_train, opt, crit, sc, device,
                    accum_steps=ACCUM_STEPS, scheduler=sch)
        _,va,*_ = eval_epoch(m, fl_val, crit, device)
        if va > best_acc:
            best_acc=va; torch.save(m.state_dict(), ckpt); pat=0
        else:
            pat+=1
            if pat>=KF_PATIENCE: print(f'  Early stop ep{ep}'); break
        print(f'  ep{ep:02d}: val_acc={va:.4f}')

    m.load_state_dict(torch.load(ckpt, map_location=device))
    fold_models.append(ckpt)

    # TTA evaluation on fold test
    print(f'  Running TTA ({len(TTA_TRANSFORMS)} passes)...')
    tta_preds, tta_labs, tta_probs = tta_eval(m, fl_test, device)
    ta   = accuracy_score(tta_labs, tta_preds)
    f1_f = f1_score(tta_labs, tta_preds, average='weighted', zero_division=0)
    prec = precision_score(tta_labs, tta_preds, average='weighted', zero_division=0)
    rec  = recall_score(tta_labs, tta_preds, average='weighted', zero_division=0)
    try: tauc = roc_auc_score(tta_labs, tta_probs)
    except ValueError: tauc = float('nan')
    _,_,tsa,_,_,_,_,_,_,_ = eval_epoch(m, fl_test, crit, device)

    fold_results.append({'fold':fold+1,'val_acc':best_acc,
        'test_acc':ta,'sub_acc':tsa,'auc':tauc,
        'f1':f1_f,'precision':prec,'recall':rec,
        'preds':tta_preds,'labels':tta_labs,'probs':tta_probs})
    print(f'  → TTA acc={ta:.4f}  sub={tsa:.4f}  auc={tauc:.4f}  f1={f1_f:.4f}')
    del m; torch.cuda.empty_cache()

# Aggregate
accs  = [r['test_acc']  for r in fold_results]
aucs  = [r['auc']       for r in fold_results]
f1s   = [r['f1']        for r in fold_results]
precs = [r['precision'] for r in fold_results]
recs  = [r['recall']    for r in fold_results]
subs  = [r['sub_acc']   for r in fold_results]

acc_m,  acc_h  = ci95(accs)
auc_m,  auc_h  = ci95(aucs)
f1_m,   f1_h   = ci95(f1s)
prec_m, prec_h = ci95(precs)
rec_m,  rec_h  = ci95(recs)
sub_m,  sub_h  = ci95(subs)

print('\n' + '='*65)
print(f'{N_FOLDS}-Fold CV + TTA  (mean ± 95% CI, t-dist df={N_FOLDS-1})')
print('='*65)
print(f'Binary Accuracy : {acc_m:.4f} ± {acc_h:.4f}')
print(f'AUC-ROC         : {auc_m:.4f} ± {auc_h:.4f}')
print(f'F1 (weighted)   : {f1_m:.4f}  ± {f1_h:.4f}')
print(f'Precision       : {prec_m:.4f} ± {prec_h:.4f}')
print(f'Recall          : {rec_m:.4f}  ± {rec_h:.4f}')
print(f'Subtype Acc     : {sub_m:.4f}  ± {sub_h:.4f}  (auxiliary)')
print('='*65)

kfold_df = pd.DataFrame(fold_results).drop(columns=['preds','labels','probs'])
kfold_df.to_csv('results/kfold_results.csv', index=False)
ci_summary = {
    'n_folds':N_FOLDS,'n_patients':len(all_patients),
    'evaluation':'5-fold CV + TTA (5 transforms)',
    'binary_acc' :{'mean':round(acc_m,4), 'ci95':round(acc_h,4)},
    'auc_roc'    :{'mean':round(auc_m,4), 'ci95':round(auc_h,4)},
    'f1_weighted':{'mean':round(f1_m,4),  'ci95':round(f1_h,4)},
    'precision'  :{'mean':round(prec_m,4),'ci95':round(prec_h,4)},
    'recall'     :{'mean':round(rec_m,4), 'ci95':round(rec_h,4)},
    'subtype_acc':{'mean':round(sub_m,4), 'ci95':round(sub_h,4)},
}
with open('results/cv_confidence_intervals.json','w') as f:
    json.dump(ci_summary, f, indent=2)
print('Saved: results/kfold_results.csv + cv_confidence_intervals.json')


### 7.1 CV Visualisation

In [ ]:
# Pooled predictions across all folds
all_fold_preds  = [p for r in fold_results for p in r['preds']]
all_fold_labels = [l for r in fold_results for l in r['labels']]
all_fold_probs  = [p for r in fold_results for p in r['probs']]

fig, axes = plt.subplots(1,3,figsize=(16,5))

# Per-fold metrics
x = range(1, N_FOLDS+1)
axes[0].bar(x, accs, color='steelblue', alpha=0.8, label='Accuracy')
axes[0].bar(x, aucs, color='tomato',    alpha=0.8, label='AUC', bottom=0)
axes[0].axhline(acc_m, color='navy',   linestyle='--', label=f'Mean Acc={acc_m:.3f}')
axes[0].axhline(auc_m, color='darkred',linestyle='--', label=f'Mean AUC={auc_m:.3f}')
axes[0].set_title('Per-Fold Performance'); axes[0].set_xlabel('Fold')
axes[0].set_ylim(0,1.1); axes[0].legend(fontsize=8)

# Pooled confusion matrix
cm_cv = confusion_matrix(all_fold_labels, all_fold_preds)
sns.heatmap(cm_cv, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Benign','Malignant'], yticklabels=['Benign','Malignant'])
axes[1].set_title('Pooled CV Confusion Matrix')

# ROC on pooled predictions
fpr_cv, tpr_cv, _ = roc_curve(all_fold_labels, all_fold_probs)
auc_pooled = roc_auc_score(all_fold_labels, all_fold_probs)
axes[2].plot(fpr_cv, tpr_cv, lw=2, label=f'Pooled AUC={auc_pooled:.4f}')
axes[2].plot([0,1],[0,1],'k--')
axes[2].set_xlabel('FPR'); axes[2].set_ylabel('TPR')
axes[2].set_title('Pooled CV ROC Curve'); axes[2].legend()

plt.suptitle(f'{N_FOLDS}-Fold Patient-wise CV Results', fontsize=13)
plt.tight_layout()
plt.savefig('plots/cv_results.png', dpi=150)
plt.show()

### 7.2 5-Fold Model Ensemble (Phase 4.1)

Load all 5 fold checkpoints and soft-vote on the fixed test split.  
Ensemble reduces variance and typically lifts accuracy 2-3% over single fold.

In [ ]:
# ── Phase 4.1: 5-fold ensemble on fixed test split ───────────────────────────
# Load all fold models, average their softmax probabilities.

def ensemble_predict(fold_ckpts, loader, device):
    """
    Soft-voting ensemble across all saved fold models.
    Averages P(malignant) across folds, then thresholds at 0.5.
    """
    all_run_probs = []
    all_labs      = []

    for ckpt in fold_ckpts:
        if not os.path.exists(ckpt):
            print(f'  Skipping missing checkpoint: {ckpt}'); continue
        torch.cuda.empty_cache()
        m = MAGNET(feat_dim=FEAT_DIM, pretrained=False,
                   use_gnn=True, use_aux=True).to(device)
        m.load_state_dict(torch.load(ckpt, map_location=device))
        m.eval()
        run_probs = []; run_labs = []
        with torch.no_grad():
            for batch in loader:
                x40,x100,x200,x400,y_bin,_ = [b.to(device) for b in batch]
                with autocast_ctx():
                    bl,*_ = m(x40,x100,x200,x400)
                run_probs.extend(F.softmax(bl.float(),1)[:,1].cpu().tolist())
                run_labs.extend(y_bin.cpu().tolist())
        all_run_probs.append(run_probs)
        all_labs = run_labs
        del m; torch.cuda.empty_cache()

    avg_probs = np.array(all_run_probs).mean(axis=0)
    preds     = (avg_probs >= 0.5).astype(int).tolist()
    return preds, all_labs, avg_probs.tolist()


ensemble_preds, ensemble_labs, ensemble_probs = ensemble_predict(
    fold_models, test_loader, device)

ens_acc  = accuracy_score(ensemble_labs, ensemble_preds)
ens_f1   = f1_score(ensemble_labs, ensemble_preds, average='weighted', zero_division=0)
try: ens_auc = roc_auc_score(ensemble_labs, ensemble_probs)
except ValueError: ens_auc = float('nan')

print('='*55)
print(f'5-Fold Ensemble — Fixed Test Split (n={len(ensemble_labs)})')
print(f'  Accuracy : {ens_acc:.4f}')
print(f'  AUC-ROC  : {ens_auc:.4f}')
print(f'  F1       : {ens_f1:.4f}')
print('='*55)
print(classification_report(ensemble_labs, ensemble_preds,
                             target_names=['Benign','Malignant']))

# Compare ensemble vs single best model on fixed test
print(f'Single best model (fixed split): acc={test_acc:.4f}')
print(f'5-fold ensemble (fixed split)  : acc={ens_acc:.4f}  Δ={ens_acc-test_acc:+.4f}')

# Save ensemble results
with open('results/all_results.json') as f:
    all_res = json.load(f)
all_res['ensemble_fixed_split'] = {
    'accuracy': round(float(ens_acc),4),
    'auc_roc' : round(float(ens_auc),4),
    'f1'      : round(float(ens_f1),4),
    'n_models': len(fold_models),
}
with open('results/all_results.json','w') as f:
    json.dump(all_res, f, indent=2)
print('Saved ensemble results to results/all_results.json')


## 8. Statistical Significance Tests

In [ ]:
def run_mcnemar(preds_a, preds_b, labels, name_a='MAGNET', name_b='Baseline'):
    """
    McNemar's exact test.
    H0: both models have the same per-sample error rate.
    """
    labels  = np.array(labels)
    preds_a = np.array(preds_a)
    preds_b = np.array(preds_b)
    ca = (preds_a == labels)
    cb = (preds_b == labels)
    b  = (ca & ~cb).sum()   # A right, B wrong
    c  = (~ca & cb).sum()   # A wrong, B right
    table = [[int((ca & cb).sum()), int(b)],
             [int(c),               int((~ca & ~cb).sum())]]
    res = mcnemar_test_fn(table, exact=True)
    sig = 'p<0.05 ✓' if res.pvalue < 0.05 else 'not significant'
    print(f'McNemar ({name_a} vs {name_b}): b={b} c={c}  p={res.pvalue:.4f}  [{sig}]')
    return res.pvalue

# Majority-class baseline (always predict malignant = 1)
majority_preds = [1]*len(all_fold_labels)
p_majority = run_mcnemar(all_fold_preds, majority_preds, all_fold_labels,
                         'MAGNET','Majority-class')

# Proportion test with 95% CI on pooled CV accuracy
n_correct = sum(p==l for p,l in zip(all_fold_preds, all_fold_labels))
n_total   = len(all_fold_labels)
prop      = n_correct / n_total
prop_ci   = st.norm.ppf(0.975) * np.sqrt(prop*(1-prop)/n_total)
print(f'\nPooled CV accuracy: {prop:.4f} ± {prop_ci:.4f}  (95% CI, n={n_total} samples)')

mcnemar_results = {
    'vs_majority_class': {'p_value': float(p_majority), 'significant_at_0.05': bool(p_majority < 0.05)},
    'pooled_accuracy'  : {'mean': round(float(prop),4), 'ci95': round(float(prop_ci),4), 'n': int(n_total)}
}
with open('results/statistical_tests.json','w') as f:
    json.dump(mcnemar_results, f, indent=2)
print('Saved: results/statistical_tests.json')

### 8.2 McNemar Tests: MAGNET vs Each Baseline

In [ ]:
# ── McNemar: MAGNET vs each baseline on fixed test split ─────────────────────
# Reload each saved baseline checkpoint and evaluate on the same test patients.
# All loaders use the MIL format (n_per_mag=IMAGES_PER_MAG, augment=False).

def get_baseline_preds_mil(backbone_name, mag):
    """
    Reload a saved single-backbone MIL baseline and get test-set predictions.
    Uses SingleMagDataset (MIL version, augment=False, n_per_mag=IMAGES_PER_MAG).
    """
    ckpt = f'checkpoints/baseline_{_safe_name(backbone_name)}_{mag}.pth'
    if not os.path.exists(ckpt):
        print(f'  Checkpoint not found: {ckpt} — skipping'); return None, None

    # Rebuild same architecture as train_single_baseline
    backbone  = timm.create_model(backbone_name, pretrained=False,
                                   num_classes=0, global_pool='avg').to(device)
    # Infer in_dim from backbone
    with torch.no_grad():
        in_dim = backbone(torch.zeros(1,3,IMG_SIZE,IMG_SIZE).to(device)).shape[1]
    attn_pool  = nn.Sequential(nn.Linear(in_dim,128), nn.Tanh(),
                                nn.Linear(128,1)).to(device)
    classifier = nn.Sequential(nn.Linear(in_dim,256), nn.BatchNorm1d(256),
                                nn.GELU(), nn.Dropout(0.4),
                                nn.Linear(256,2)).to(device)
    ckpt_d = torch.load(ckpt, map_location=device)
    backbone.load_state_dict(ckpt_d['bb'])
    attn_pool.load_state_dict(ckpt_d['attn'])
    classifier.load_state_dict(ckpt_d['cls'])
    backbone.eval(); attn_pool.eval(); classifier.eval()

    # MIL test loader — same format as training
    ds = SingleMagDataset(test_patients, patient_info, mag,
                          augment=False, img_size=IMG_SIZE,
                          n_per_mag=IMAGES_PER_MAG)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=False)

    preds, labs = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            B, N, C, H, W = imgs.shape
            imgs = imgs.to(device)
            feats  = backbone(imgs.view(B*N,C,H,W)).view(B,N,-1)
            attn   = torch.softmax(attn_pool(feats), dim=1)
            pooled = (attn * feats).sum(1)
            preds.extend(classifier(pooled).argmax(1).cpu().tolist())
            labs.extend(labels.tolist())
    del backbone, attn_pool, classifier; torch.cuda.empty_cache()
    return preds, labs


def get_concat_preds_mil():
    """
    Reload saved ConcatFusionBaseline (MIL version) and evaluate on test set.
    Uses the multi-mag MIL loader.
    """
    ckpt = 'checkpoints/concat_baseline.pth'
    if not os.path.exists(ckpt):
        print(f'  Checkpoint not found: {ckpt} — skipping'); return None, None
    m = ConcatFusionBaseline(pretrained=False).to(device)
    m.load_state_dict(torch.load(ckpt, map_location=device))
    m.eval()
    preds, labs = [], []
    with torch.no_grad():
        for batch in test_loader:   # test_loader is already MIL [B,N,C,H,W]
            x40,x100,x200,x400,y_bin,_ = [b.to(device) for b in batch]
            preds.extend(m(x40,x100,x200,x400).argmax(1).cpu().tolist())
            labs.extend(y_bin.cpu().tolist())
    del m; torch.cuda.empty_cache()
    return preds, labs


print('McNemar tests: MAGNET vs each baseline')
print('='*55)
mcnemar_vs_baselines = {}

baseline_eval_configs = [
    ('resnet34',             '40X'),
    ('efficientnet_b0',      '40X'),
    ('mobilenetv3_large_100','40X'),
    ('resnet34',             '200X'),
]
for bn, mag in baseline_eval_configs:
    bp, bl = get_baseline_preds_mil(bn, mag)
    if bp is None: continue
    name = f'{bn}@{mag}'
    pval = run_mcnemar(test_preds, bp, test_labels, 'MAGNET', name)
    mcnemar_vs_baselines[name] = {'p_value': float(pval),
                                   'significant': bool(pval < 0.05)}

cp, cl = get_concat_preds_mil()
if cp is not None:
    pval = run_mcnemar(test_preds, cp, test_labels, 'MAGNET', 'ConcatFusion')
    mcnemar_vs_baselines['ConcatFusion'] = {'p_value': float(pval),
                                              'significant': bool(pval < 0.05)}

# Update statistical_tests.json
with open('results/statistical_tests.json') as f:
    stat_tests = json.load(f)
stat_tests['vs_baselines'] = mcnemar_vs_baselines
with open('results/statistical_tests.json','w') as f:
    json.dump(stat_tests, f, indent=2)

print('\nMcNemar summary:')
for name, res in mcnemar_vs_baselines.items():
    sig = '✓ significant' if res['significant'] else '✗ not significant'
    print(f'  MAGNET vs {name:<35}: p={res["p_value"]:.4f}  {sig}')
print('Saved: results/statistical_tests.json')


### 8.3 Cross-Validation for Best Baseline

Runs 5-fold CV on the best single-backbone baseline (EfficientNet-B0 @ 40X)  
so MAGNET's CV results can be compared directly against a baseline CV estimate.

In [ ]:
# ── 5-fold CV for best baseline: EfficientNet-B0 @ 40X ───────────────────────
# Uses the same folds as MAGNET CV (same StratifiedKFold seed) for fair comparison.

BASELINE_CV_BACKBONE = 'efficientnet_b0'
BASELINE_CV_MAG      = '40X'
BASELINE_CV_EPOCHS   = 20
BASELINE_CV_PATIENCE = 6

baseline_cv_results = []

print(f'5-Fold CV: {BASELINE_CV_BACKBONE} @ {BASELINE_CV_MAG}')
print('='*55)

for fold, (train_idx, test_idx) in enumerate(
        StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42
                        ).split(all_patients, pat_labels)):

    fold_train_all = [all_patients[i] for i in train_idx]
    fold_test      = [all_patients[i] for i in test_idx]
    n_val          = max(2, int(0.15 * len(fold_train_all)))
    fold_val       = fold_train_all[-n_val:]
    fold_train     = fold_train_all[:-n_val]

    def _smloader(patients, tf, sh):
        ds = SingleMagDataset(patients, patient_info, BASELINE_CV_MAG,
                              transform=tf, img_size=IMG_SIZE)
        return DataLoader(ds, batch_size=BATCH_SIZE*2, shuffle=sh,
                          num_workers=NUM_WORKERS, pin_memory=False, drop_last=sh)

    bl_train = _smloader(fold_train, train_tf, True)
    bl_val   = _smloader(fold_val,   val_tf,  False)
    bl_test  = _smloader(fold_test,  val_tf,  False)

    print(f'\nFold {fold+1}/{N_FOLDS} — train={len(fold_train)} val={len(fold_val)} test={len(fold_test)}')
    torch.cuda.empty_cache()
    m = timm.create_model(BASELINE_CV_BACKBONE, pretrained=True,
                           num_classes=2, global_pool='avg').to(device)
    opt = optim.AdamW(m.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=BASELINE_CV_EPOCHS)
    ce  = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    sc  = make_scaler()
    best_acc=0.0; pat=0
    ckpt = f'checkpoints/baseline_cv_fold_{fold+1}.pth'

    for ep in range(1, BASELINE_CV_EPOCHS+1):
        m.train()
        for imgs, labels in bl_train:
            imgs, labels = imgs.to(device), labels.to(device)
            opt.zero_grad()
            with autocast_ctx(): loss = ce(m(imgs), labels)
            sc.scale(loss).backward()
            sc.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            sc.step(opt); sc.update()
        sch.step()
        m.eval(); preds, labs = [], []
        with torch.no_grad():
            for imgs, labels in bl_val:
                preds.extend(m(imgs.to(device)).argmax(1).cpu().tolist())
                labs.extend(labels.tolist())
        va = accuracy_score(labs, preds)
        if va > best_acc: best_acc=va; torch.save(m.state_dict(), ckpt); pat=0
        else:
            pat+=1
            if pat >= BASELINE_CV_PATIENCE: break
        print(f'  ep{ep:02d}: val_acc={va:.4f}')

    m.load_state_dict(torch.load(ckpt, map_location=device)); m.eval()
    preds, labs, probs = [], [], []
    with torch.no_grad():
        for imgs, labels in bl_test:
            logits = m(imgs.to(device))
            preds.extend(logits.argmax(1).cpu().tolist())
            probs.extend(F.softmax(logits,1)[:,1].cpu().tolist())
            labs.extend(labels.tolist())
    ta  = accuracy_score(labs, preds)
    f1b = f1_score(labs, preds, average='weighted', zero_division=0)
    try: ta_auc = roc_auc_score(labs, probs)
    except ValueError: ta_auc = float('nan')
    baseline_cv_results.append({'fold':fold+1,'test_acc':ta,'auc':ta_auc,'f1':f1b})
    print(f'  → acc={ta:.4f}  auc={ta_auc:.4f}  f1={f1b:.4f}')
    del m; torch.cuda.empty_cache()

# ── Aggregate + compare ───────────────────────────────────────────────────────
bl_accs = [r['test_acc'] for r in baseline_cv_results]
bl_aucs = [r['auc']      for r in baseline_cv_results]
bl_f1s  = [r['f1']       for r in baseline_cv_results]
bl_acc_m, bl_acc_h = ci95(bl_accs)
bl_auc_m, bl_auc_h = ci95(bl_aucs)
bl_f1_m,  bl_f1_h  = ci95(bl_f1s)

print('\n' + '='*55)
print(f'Baseline CV ({BASELINE_CV_BACKBONE}@{BASELINE_CV_MAG}):')
print(f'  Accuracy : {bl_acc_m:.4f} ± {bl_acc_h:.4f}')
print(f'  AUC-ROC  : {bl_auc_m:.4f} ± {bl_auc_h:.4f}')
print(f'  F1       : {bl_f1_m:.4f}  ± {bl_f1_h:.4f}')
print('\nMAGNET CV (from Section 7):')
print(f'  Accuracy : {acc_m:.4f} ± {acc_h:.4f}')
print(f'  AUC-ROC  : {auc_m:.4f} ± {auc_h:.4f}')
print(f'  F1       : {f1_m:.4f}  ± {f1_h:.4f}')

# Check if CI intervals overlap
acc_overlap = (acc_m - acc_h) < (bl_acc_m + bl_acc_h)
print(f'\nCI ranges overlap (acc): {acc_overlap} '
      f'[MAGNET: {acc_m-acc_h:.4f}–{acc_m+acc_h:.4f}, '
      f'Baseline: {bl_acc_m-bl_acc_h:.4f}–{bl_acc_m+bl_acc_h:.4f}]')

pd.DataFrame(baseline_cv_results).to_csv('results/baseline_cv_results.csv', index=False)

# Save to CI summary
ci_summary['best_baseline_cv'] = {
    'model': f'{BASELINE_CV_BACKBONE}@{BASELINE_CV_MAG}',
    'binary_acc': {'mean': round(bl_acc_m,4), 'ci95': round(bl_acc_h,4)},
    'auc_roc':    {'mean': round(bl_auc_m,4), 'ci95': round(bl_auc_h,4)},
    'f1_weighted':{'mean': round(bl_f1_m,4),  'ci95': round(bl_f1_h,4)},
}
with open('results/cv_confidence_intervals.json','w') as f:
    json.dump(ci_summary, f, indent=2)
print('Saved: results/baseline_cv_results.csv')


## 9. Ablation Studies

In [ ]:
ABLATION_EPOCHS   = 20
ABLATION_PATIENCE = 8

def run_ablation(config_name, use_gnn=True, use_aux=True, use_attn=True):
    print(f'\nAblation: {config_name}')
    torch.cuda.empty_cache()
    m = MAGNET(feat_dim=FEAT_DIM, pretrained=True, use_gnn=use_gnn, use_aux=use_aux).to(device)

    if not use_attn:
        import types
        def mean_fwd(self, feats):
            x = torch.stack(feats,dim=1)
            return x.mean(1), torch.zeros(x.size(0),4,4,device=x.device)
        m.fusion.forward = types.MethodType(mean_fwd, m.fusion)

    bb_p = [p for mg in MAGNIFICATIONS for p in m.branches[mg].backbone.parameters()]
    oth  = [p for n,p in m.named_parameters()
            if not any(n.startswith(f'branches.{mg}.backbone') for mg in MAGNIFICATIONS)]
    opt  = optim.AdamW([{'params':bb_p,'lr':LR*0.1},{'params':oth,'lr':LR}], weight_decay=WEIGHT_DECAY)
    sch  = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=ABLATION_EPOCHS)
    crit = MAGNETLoss(gamma=0.3 if use_aux else 0.0)
    sc   = make_scaler()

    best_acc=0.0; pat=0
    ckpt = os.path.join('checkpoints', f'ablation_{_safe_name(config_name)}.pth')

    for ep in range(1, ABLATION_EPOCHS+1):
        train_epoch(m, train_loader, opt, crit, sc, device)
        _,va,*_ = eval_epoch(m, val_loader, crit, device)
        sch.step()
        if va>best_acc: best_acc=va; torch.save(m.state_dict(),ckpt); pat=0
        else:
            pat+=1
            if pat>=ABLATION_PATIENCE: print(f'  Early stop ep{ep}'); break
        print(f'  ep{ep:02d}: val_acc={va:.4f}')

    m.load_state_dict(torch.load(ckpt, map_location=device))
    _,ta,tsa,tauc,tp,tl,_,_,_,_ = eval_epoch(m, test_loader, crit, device)
    f1a = f1_score(tl,tp,average='weighted',zero_division=0)
    del m; torch.cuda.empty_cache()
    print(f'  → acc={ta:.4f} sub={tsa:.4f} auc={tauc:.4f} f1={f1a:.4f}')
    return {'config':config_name,'val_acc':best_val_acc,
            'test_acc':ta,'sub_acc':tsa,'auc':tauc,'f1':f1a}

ablation_rows = [{
    'config':'MAGNET (full)','val_acc':best_val_acc,
    'test_acc':test_acc,'sub_acc':test_sub_acc,'auc':test_auc,
    'f1':f1_score(test_labels,test_preds,average='weighted',zero_division=0)
}]
ablation_rows.append(run_ablation('wo_GNN',  use_gnn=False, use_aux=True,  use_attn=True))
ablation_rows.append(run_ablation('wo_Aux',  use_gnn=True,  use_aux=False, use_attn=True))
ablation_rows.append(run_ablation('wo_Attn', use_gnn=True,  use_aux=True,  use_attn=False))
ablation_rows.append(run_ablation('wo_Both', use_gnn=False, use_aux=False, use_attn=False))

label_map = {'wo_GNN':'w/o GNN','wo_Aux':'w/o Aux','wo_Attn':'w/o Attn','wo_Both':'w/o GNN+Aux+Attn'}
for r in ablation_rows: r['config'] = label_map.get(r['config'], r['config'])

ablation_df = pd.DataFrame(ablation_rows)
ablation_df.to_csv('results/ablation_results.csv', index=False)
print('\n' + ablation_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(11,5))
x = range(len(ablation_df)); w=0.25
ax.bar([i-w for i in x], ablation_df['test_acc'], w, label='Test Acc')
ax.bar([i   for i in x], ablation_df['auc'],      w, label='AUC-ROC')
ax.bar([i+w for i in x], ablation_df['f1'],       w, label='F1')
ax.set_xticks(list(x)); ax.set_xticklabels(ablation_df['config'], rotation=15)
ax.set_ylim(0,1.1); ax.legend(); ax.set_title('Ablation Study')
plt.tight_layout(); plt.savefig('plots/ablation.png',dpi=150); plt.show()

## 10. Baseline Comparisons

In [ ]:
# ── Single-backbone baseline training calls ──────────────────────────────────
# Class definitions (SingleMagDataset, train_single_baseline) are in Section 3.3

baseline_rows = []
baseline_rows.append(train_single_baseline('resnet34',             '40X'))
baseline_rows.append(train_single_baseline('efficientnet_b0',      '40X'))
baseline_rows.append(train_single_baseline('mobilenetv3_large_100','40X'))
baseline_rows.append(train_single_baseline('resnet34',             '200X'))
print('\nAll single-backbone baselines done.')


In [ ]:
# ── ConcatFusion baseline training call ──────────────────────────────────────
# Class definition (ConcatFusionBaseline, run_concat_baseline) in Section 3.3

baseline_rows.append(run_concat_baseline())
baseline_rows.append({
    'config':'MAGNET (ours)',
    'test_acc': test_acc,
    'auc'     : test_auc,
    'f1'      : f1_score(test_labels, test_preds, average='weighted', zero_division=0)
})

baseline_df = pd.DataFrame(baseline_rows)
baseline_df.to_csv('results/baseline_comparison.csv', index=False)
print('\n' + baseline_df.to_string(index=False))


In [ ]:
# ── SOTA Comparison Table ─────────────────────────────────────────────────────
sota_df = pd.DataFrame([
    {'Method':'Spanhol et al. 2016 [CNN]',          'Acc':0.805, 'Split':'image-level','Notes':'Single mag'},
    {'Method':'Han et al. 2017 [CNN+Transfer]',     'Acc':0.880, 'Split':'image-level','Notes':'Multi-task'},
    {'Method':'Alom et al. 2019 [IRRCNN]',          'Acc':0.905, 'Split':'image-level','Notes':'Recurrent CNN'},
    {'Method':'Shu et al. 2020 [Multi-mag CNN]',    'Acc':0.910, 'Split':'image-level','Notes':'Multi-mag'},
    {'Method':'Zhu et al. 2021 [GNN]',              'Acc':0.870, 'Split':'patient-level','Notes':'Graph-based'},
    {'Method':f'MAGNET (ours, CV mean±CI)',
     'Acc':acc_m, 'Split':'patient-level',
     'Notes':f'±{acc_h:.4f} (95% CI, {N_FOLDS}-fold)'},
])
sota_df.to_csv('results/sota_comparison.csv',index=False)
print('SOTA Comparison:')
print(sota_df.to_string(index=False))
print('\n⚠ Image-level splits are NOT directly comparable to patient-level splits.')
print('  Patient-level splitting prevents data leakage and is methodologically stricter.')
print('  Most literature uses image-level splits, inflating reported accuracy.')

In [ ]:
# Comparison bar chart
fig, ax = plt.subplots(figsize=(13,5))
colors  = ['#4C72B0']*len(baseline_df); colors[-1]='#DD8452'
bars = ax.barh(baseline_df['config'], baseline_df['test_acc'], color=colors)
ax.set_xlabel('Test Accuracy'); ax.set_title('Baseline vs MAGNET (fixed split)')
ax.set_xlim(0,1.1)
for bar,val in zip(bars,baseline_df['test_acc']):
    ax.text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)
plt.tight_layout(); plt.savefig('plots/baseline_comparison.png',dpi=150); plt.show()

## 11. Interpretability

### 11.1 Attention Weight Visualisation

In [ ]:
model.load_state_dict(torch.load('checkpoints/best_model.pth', map_location=device))
model.eval()

all_attn = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Attention'):
        x40,x100,x200,x400,_,_ = [b.to(device) for b in batch]
        with autocast_ctx():
            *_, aw = model(x40,x100,x200,x400,return_attn=True)
        all_attn.append(aw.cpu().float())

all_attn  = torch.cat(all_attn, dim=0)   # [N,4,4]
mean_attn = all_attn.mean(0).numpy()

fig, axes = plt.subplots(1,2,figsize=(12,5))
sns.heatmap(mean_attn, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=MAGNIFICATIONS, yticklabels=MAGNIFICATIONS, ax=axes[0])
axes[0].set_title('Mean Attention Weights (Test Set)')

# Softmax learnable scale weights
scale_w = F.softmax(model.fusion.mag_scale.detach().cpu(), dim=0).numpy()
axes[1].bar(MAGNIFICATIONS, scale_w, color='steelblue')
axes[1].set_title('Learned Magnification Scale Weights')
axes[1].set_ylabel('Softmax weight')

plt.tight_layout(); plt.savefig('plots/attention_analysis.png',dpi=150); plt.show()

print('Diagonal (self-attention weight per magnification):')
for mag,w in zip(MAGNIFICATIONS, np.diag(mean_attn)):
    print(f'  {mag}: {w:.4f}')

### 11.2 Grad-CAM (fixed for MobileNetV3/EfficientNet inplace Hardswish)

In [ ]:
class GradCAM:
    """
    Grad-CAM fixed for MobileNetV3 / EfficientNet inplace Hardswish/SiLU.
    Hooks on the last feature-map conv BEFORE forward_head (avoids inplace activation).
    Clones both activation and gradient tensors immediately on save.
    """
    def __init__(self, model, target_layer):
        self.model=model; self.activations=None; self.gradients=None
        self._fwd = target_layer.register_forward_hook(self._save_act)
        self._bwd = target_layer.register_full_backward_hook(self._save_grad)

    def _save_act(self, m, inp, out): self.activations = out.detach().clone()
    def _save_grad(self, m, gi, go): self.gradients    = go[0].detach().clone()

    def __call__(self, x40, x100, x200, x400, class_idx=None):
        self.model.zero_grad()
        bl,sl,unc,aux = self.model(x40,x100,x200,x400)
        if class_idx is None: class_idx = bl.argmax(1)[0].item()
        bl[0,class_idx].backward()
        alpha = self.gradients.mean(dim=(2,3),keepdim=True)
        cam   = (alpha*self.activations).sum(1,keepdim=True)
        cam   = F.relu(cam)
        cam   = F.interpolate(cam,(224,224),mode='bilinear',align_corners=False)
        cam   = cam.squeeze().cpu().float().numpy()
        if cam.max()>0: cam=cam/cam.max()
        return cam, class_idx

    def remove_hooks(self): self._fwd.remove(); self._bwd.remove()


def get_gradcam_layer(branch_extractor, mag):
    """
    Returns the last safe Conv2d for Grad-CAM — before any inplace head activation.
    ResNet34  : layer4[-1].conv2  (followed by non-inplace BN+ReLU)
    MobileNetV3/EfficientNet: last conv_pwl in blocks[-1][-1]
                              (followed by BN with act=Identity)
    """
    bb = branch_extractor.backbone
    if hasattr(bb,'layer4'):
        return bb.layer4[-1].conv2
    if hasattr(bb,'blocks'):
        last_block = list(bb.blocks[-1].children())[-1]
        for attr in ['conv_pwl','conv_pw','conv']:
            layer = getattr(last_block, attr, None)
            if isinstance(layer, nn.Conv2d): return layer
    # Fallback: second-to-last Conv2d (skip conv_head which feeds Hardswish)
    all_convs = [m for m in bb.modules() if isinstance(m,nn.Conv2d)]
    return all_convs[-2] if len(all_convs)>=2 else (all_convs[-1] if all_convs else None)


def show_gradcam(model, loader, device, n_samples=2, save_path='plots/gradcam.png'):
    model.eval()
    batch = next(iter(loader))
    x40,x100,x200,x400,y_bin,_ = [b.to(device) for b in batch]
    n_samples = min(n_samples, x40.shape[0])  # guard: batch may be smaller
    # MIL format: [B,N,C,H,W] — use only the first image (N=0) per mag.
    # GradCAM needs a 4-D spatial tensor; N=1 slice gives [n_samples,1,C,H,W]
    def _slice(x): return x[:n_samples, 0:1]   # [n_samples, 1, C, H, W]
    inputs={'40X':_slice(x40),'100X':_slice(x100),
            '200X':_slice(x200),'400X':_slice(x400)}

    inv_mean = [-m/s for m,s in zip(IMG_MEAN,IMG_STD)]
    inv_std  = [1/s for s in IMG_STD]
    from torchvision import transforms as T
    denorm = T.Normalize(inv_mean,inv_std)

    fig,axes = plt.subplots(n_samples, len(MAGNIFICATIONS)+1,
                             figsize=(5*(len(MAGNIFICATIONS)+1), 4*n_samples))
    if n_samples==1: axes=[axes]

    for si in range(n_samples):
        row = axes[si]
        img40 = denorm(x40[si, 0].cpu()).permute(1,2,0).clamp(0,1).numpy()
        row[0].imshow(img40)
        row[0].set_title(f'40X\n(true={y_bin[si].item()})')
        row[0].axis('off')

        for cj, mag in enumerate(MAGNIFICATIONS, start=1):
            tl = get_gradcam_layer(model.branches[mag], mag)
            if tl is None:
                row[cj].text(0.5,0.5,'No layer',ha='center'); row[cj].axis('off'); continue
            single = {m: inputs[m][si:si+1] for m in MAGNIFICATIONS}
            model.train()
            cam_gen = GradCAM(model, tl)
            try:
                cam,pred = cam_gen(single['40X'],single['100X'],single['200X'],single['400X'])
                img_m    = denorm(inputs[mag][si, 0].cpu()).permute(1,2,0).clamp(0,1).numpy()
                ov       = cv2.applyColorMap((cam*255).astype(np.uint8),cv2.COLORMAP_JET)
                ov       = cv2.cvtColor(ov,cv2.COLOR_BGR2RGB)/255.0
                row[cj].imshow((0.5*img_m+0.5*ov).clip(0,1))
                row[cj].set_title(f'{mag}\n(pred={pred})')
            except Exception as e:
                row[cj].text(0.5,0.5,f'Error:\n{str(e)[:40]}',ha='center',fontsize=6)
                row[cj].set_title(f'{mag} — failed')
            finally:
                cam_gen.remove_hooks(); model.eval()
            row[cj].axis('off')

    plt.suptitle('Grad-CAM per Backbone Branch',fontsize=13)
    plt.tight_layout()
    plt.savefig(save_path,dpi=150,bbox_inches='tight'); plt.show()
    print(f'Saved: {save_path}')

model.train(); show_gradcam(model, test_loader, device); model.eval()

### 11.3 Monte-Carlo Dropout Uncertainty

In [ ]:
def mc_uncertainty_analysis(model, loader, device, n_passes=20, max_batches=5):
    all_means=[]; all_vars=[]; all_labels=[]; all_preds=[]
    for i,batch in enumerate(loader):
        if i>=max_batches: break
        x40,x100,x200,x400,y_bin,_ = [b.to(device) for b in batch]
        # MIL format: x40 is [B,N,C,H,W]; BN check on batch dim
        if x40.shape[0]<2:
            print(f'  Skipping singleton batch {i} (BN requires ≥2 samples)'); continue
        mean_p, var_p = model.mc_uncertainty(x40,x100,x200,x400,n_passes)
        all_means.append(mean_p.cpu())
        all_vars.append(var_p.cpu())
        all_labels.extend(y_bin.cpu().tolist())
        all_preds.extend(mean_p.argmax(1).cpu().tolist())

    if not all_means:
        print('No valid batches for MC analysis.'); return np.array([]),np.array([])

    all_means = torch.cat(all_means,0)
    all_vars  = torch.cat(all_vars,0)
    unc_scores= all_vars.mean(1).numpy()
    correct   = np.array(all_labels)==np.array(all_preds)

    print(f'MC-Dropout ({n_passes} passes, {len(all_labels)} samples):')
    if correct.sum()>0:  print(f'  Uncertainty (correct)  : {unc_scores[correct].mean():.6f}')
    if (~correct).sum()>0: print(f'  Uncertainty (incorrect): {unc_scores[~correct].mean():.6f}')

    plt.figure(figsize=(8,4))
    if correct.sum()>0:
        plt.hist(unc_scores[correct],  bins=15, alpha=0.6, label='Correct',   color='steelblue')
    if (~correct).sum()>0:
        plt.hist(unc_scores[~correct], bins=15, alpha=0.6, label='Incorrect', color='tomato')
    plt.xlabel('Epistemic Uncertainty (MC-Dropout Variance)')
    plt.ylabel('Frequency')
    plt.title('Uncertainty Distribution: Correct vs Misclassified')
    plt.legend(); plt.tight_layout()
    plt.savefig('plots/mc_uncertainty.png',dpi=150); plt.show()
    return unc_scores, correct

unc_scores, correct = mc_uncertainty_analysis(model, test_loader, device)

### 11.4 Uncertainty Calibration (ECE + Reliability Diagram)

Expected Calibration Error (ECE) measures whether predicted probabilities  
match empirical accuracy. A perfectly calibrated model has ECE = 0.

In [ ]:
def expected_calibration_error(probs, labels, n_bins=10):
    """
    Computes ECE and per-bin statistics for a reliability diagram.
    probs  : list/array of predicted probabilities for the positive class
    labels : list/array of true binary labels
    Returns: ece (float), bin data (list of dicts)
    """
    probs  = np.array(probs)
    labels = np.array(labels)
    bins   = np.linspace(0, 1, n_bins + 1)
    ece    = 0.0
    bin_data = []

    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        mask   = (probs >= lo) & (probs < hi)
        if i == n_bins - 1:   # include right edge in last bin
            mask = (probs >= lo) & (probs <= hi)
        n = mask.sum()
        if n == 0:
            bin_data.append({'conf': (lo+hi)/2, 'acc': 0, 'n': 0})
            continue
        avg_conf = probs[mask].mean()
        avg_acc  = labels[mask].mean()
        ece     += (n / len(probs)) * abs(avg_acc - avg_conf)
        bin_data.append({'conf': avg_conf, 'acc': avg_acc, 'n': int(n)})

    return ece, bin_data


# ── Compute ECE on pooled CV predictions (most reliable) ─────────────────────
ece_cv, bin_data_cv = expected_calibration_error(all_fold_probs, all_fold_labels)

# Also on fixed test split
ece_test, bin_data_test = expected_calibration_error(test_probs, test_labels)

print(f'Expected Calibration Error (ECE):')
print(f'  Pooled CV (n={len(all_fold_labels)}) : {ece_cv:.4f}')
print(f'  Fixed test (n={len(test_labels)})   : {ece_test:.4f}')
print('  (Lower is better; 0 = perfect calibration)')

# ── Reliability diagram ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, bd, ece_val, title in [
        (axes[0], bin_data_cv,   ece_cv,   f'Reliability Diagram — Pooled CV\nECE = {ece_cv:.4f}'),
        (axes[1], bin_data_test, ece_test, f'Reliability Diagram — Fixed Test Split\nECE = {ece_test:.4f}')]:

    confs = [b['conf'] for b in bd]
    accs  = [b['acc']  for b in bd]
    ns    = [b['n']    for b in bd]

    # Perfect calibration line
    ax.plot([0,1],[0,1],'k--',label='Perfect calibration',alpha=0.6)

    # Gap fill (over/under confidence)
    for c, a in zip(confs, accs):
        ax.fill_between([c-0.05,c+0.05],[c-0.05,c+0.05],[a-0.0,a+0.0],
                         alpha=0.15, color='tomato')

    # Bar chart of accuracy per bin
    ax.bar(confs, accs, width=0.08, alpha=0.7, color='steelblue',
           label='Model accuracy', align='center')

    # Sample count as text
    for c, a, n in zip(confs, accs, ns):
        if n > 0:
            ax.text(c, a + 0.02, str(n), ha='center', fontsize=7, color='navy')

    ax.set_xlim(0,1); ax.set_ylim(0,1.1)
    ax.set_xlabel('Mean Predicted Probability'); ax.set_ylabel('Fraction Correct')
    ax.set_title(title); ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('plots/calibration_reliability.png', dpi=150)
plt.show()
print('Saved: plots/calibration_reliability.png')

# Save ECE to results
with open('results/all_results.json') as f:
    all_res = json.load(f)
all_res['calibration'] = {
    'ece_pooled_cv'  : round(ece_cv,   4),
    'ece_fixed_test' : round(ece_test, 4),
    'n_bins'         : 10,
    'note'           : 'ECE = 0 is perfect; lower is better'
}
with open('results/all_results.json','w') as f:
    json.dump(all_res, f, indent=2)
print('ECE saved to results/all_results.json')


## 12. Save All Results

In [ ]:
final_results = {
    'model_params_M'   : round(sum(p.numel() for p in model.parameters())/1e6, 2),
    'backbone_params_M': round(sum(p.numel() for mg in MAGNIFICATIONS
                                   for p in model.branches[mg].backbone.parameters())/1e6, 2),
    'fixed_split_test' : {
        'n_patients' : len(test_patients),
        'accuracy'   : round(test_acc,4),
        'auc_roc'    : round(test_auc,4),
        'f1'         : round(f1,4),
        'precision'  : round(precision,4),
        'recall'     : round(recall,4),
        'subtype_acc': round(test_sub_acc,4),
    },
    'cross_validation' : ci_summary,
    'statistical_tests': mcnemar_results,
    'training_history' : history,
    'confusion_matrix' : confusion_matrix(test_labels,test_preds).tolist(),
    'mean_attn_weights': mean_attn.tolist(),
    'regularisation'   : {'weight_decay':WEIGHT_DECAY,'label_smoothing':LABEL_SMOOTHING},
}

with open('results/all_results.json','w') as f:
    json.dump(final_results, f, indent=2)

pd.DataFrame([final_results['fixed_split_test']]).to_csv('results/test_metrics.csv',index=False)
pd.DataFrame(classification_report(test_labels,test_preds,
    target_names=['Benign','Malignant'],output_dict=True)).T.to_csv('results/classification_report.csv')

print('Saved all results to results/')
print('\n── Primary results (use these for the paper) ──────────────────')
print(f'5-Fold CV Binary Accuracy : {acc_m:.4f} ± {acc_h:.4f}')
print(f'5-Fold CV AUC-ROC         : {auc_m:.4f} ± {auc_h:.4f}')
print(f'5-Fold CV F1 (weighted)   : {f1_m:.4f}  ± {f1_h:.4f}')
print(f'McNemar vs majority-class : p={mcnemar_results["vs_majority_class"]["p_value"]:.4f}')

## 13. LaTeX Tables for Paper

In [ ]:
# ── Table 1: CV Results ───────────────────────────────────────────────────────
t1 = '\n'.join([
    r'\begin{table}[h]\centering',
    r'\caption{5-fold patient-wise CV results on BreakHis (mean $\pm$ 95\% CI).}',
    r'\begin{tabular}{lcc}\hline',
    r'Metric & Mean & 95\% CI \\\hline',
    f'Binary Accuracy & {acc_m:.4f} & $\\pm${acc_h:.4f} \\\\',
    f'AUC-ROC & {auc_m:.4f} & $\\pm${auc_h:.4f} \\\\',
    f'F1 (weighted) & {f1_m:.4f} & $\\pm${f1_h:.4f} \\\\',
    f'Precision & {prec_m:.4f} & $\\pm${prec_h:.4f} \\\\',
    f'Recall & {rec_m:.4f} & $\\pm${rec_h:.4f} \\\\',
    r'\hline\end{tabular}\label{tab:cv}\end{table}'
])
print(t1)
with open('results/table_cv.tex','w') as f: f.write(t1)

# ── Table 2: Baseline comparison ─────────────────────────────────────────────
rows=[]
for _,r in baseline_df.iterrows():
    hi = r['config']=='MAGNET (ours)'
    a  = f'\\textbf{{{r["test_acc"]:.4f}}}' if hi else f'{r["test_acc"]:.4f}'
    u  = f'\\textbf{{{r["auc"]:.4f}}}'      if hi else f'{r["auc"]:.4f}'
    rows.append(f'{r["config"]} & {a} & {u} \\\\')
t2='\n'.join([
    r'\begin{table}[h]\centering',
    r'\caption{Comparison with baselines on BreakHis (patient-wise split).}',
    r'\begin{tabular}{lcc}\hline',
    r'Method & Accuracy & AUC-ROC \\\hline',
    *rows,
    r'\hline\end{tabular}\label{tab:baselines}\end{table}'
])
print('\n'+t2)
with open('results/table_baselines.tex','w') as f: f.write(t2)

# ── Table 3: Ablation ─────────────────────────────────────────────────────────
rows2=[]
for _,r in ablation_df.iterrows():
    hi = r['config']=='MAGNET (full)'
    a  = f'\\textbf{{{r["test_acc"]:.4f}}}' if hi else f'{r["test_acc"]:.4f}'
    u  = f'\\textbf{{{r["auc"]:.4f}}}'      if hi else f'{r["auc"]:.4f}'
    rows2.append(f'{r["config"]} & {a} & {u} & {r["f1"]:.4f} \\\\')
t3='\n'.join([
    r'\begin{table}[h]\centering',
    r'\caption{Ablation study on MAGNET components.}',
    r'\begin{tabular}{lccc}\hline',
    r'Configuration & Accuracy & AUC-ROC & F1 \\\hline',
    *rows2,
    r'\hline\end{tabular}\label{tab:ablation}\end{table}'
])
print('\n'+t3)
with open('results/table_ablation.tex','w') as f: f.write(t3)

print('\nAll LaTeX tables saved to results/')

### 13.2 LaTeX: CV Comparison Table (MAGNET vs Best Baseline)

In [ ]:
# ── Table: MAGNET vs best baseline, both with 5-fold CV ──────────────────────
t_cmp = '\n'.join([
    r'\begin{table}[h]\centering',
    r'\caption{5-fold CV comparison: MAGNET vs best single-backbone baseline '
    r'(EfficientNet-B0 at 40$\times$). Mean $\pm$ 95\% CI.}',
    r'\begin{tabular}{lccc}\hline',
    r'Method & Accuracy & AUC-ROC & F1 (wtd) \\\hline',
    f'EfficientNet-B0@40X & {bl_acc_m:.4f}$\\pm${bl_acc_h:.4f} '
    f'& {bl_auc_m:.4f}$\\pm${bl_auc_h:.4f} & {bl_f1_m:.4f}$\\pm${bl_f1_h:.4f} \\\\',
    f'\\textbf{{MAGNET (ours)}} & \\textbf{{{acc_m:.4f}$\\pm${acc_h:.4f}}} '
    f'& \\textbf{{{auc_m:.4f}$\\pm${auc_h:.4f}}} & \\textbf{{{f1_m:.4f}$\\pm${f1_h:.4f}}} \\\\',
    r'\hline\end{tabular}\label{tab:cv_compare}\end{table}'
])
print(t_cmp)
with open('results/table_cv_compare.tex','w') as f: f.write(t_cmp)
print('\nSaved: results/table_cv_compare.tex')

# ── Updated comprehensive results summary ─────────────────────────────────────
print('\n' + '='*60)
print('FINAL RESULTS SUMMARY (for manuscript)')
print('='*60)
print(f'Dataset        : BreakHis ({len(all_patients)} patients, patient-wise split)')
print(f'Evaluation     : {N_FOLDS}-fold stratified CV + statistical tests')
print()
print('Primary metric (5-fold CV, 95% CI):')
print(f'  Binary Accuracy : {acc_m:.4f} ± {acc_h:.4f}')
print(f'  AUC-ROC         : {auc_m:.4f} ± {auc_h:.4f}')
print(f'  F1 (weighted)   : {f1_m:.4f}  ± {f1_h:.4f}')
print()
print('Best baseline (EfficientNet-B0@40X, 5-fold CV):')
print(f'  Binary Accuracy : {bl_acc_m:.4f} ± {bl_acc_h:.4f}')
print(f'  AUC-ROC         : {bl_auc_m:.4f} ± {bl_auc_h:.4f}')
print()
print('Statistical significance (McNemar, exact):')
for name, res in mcnemar_vs_baselines.items():
    sig = 'p<0.05 ✓' if res['significant'] else 'n.s.'
    print(f'  MAGNET vs {name:<30}: p={res["p_value"]:.4f}  [{sig}]')
print()
print(f'Calibration (ECE, lower=better):')
print(f'  Pooled CV  : {ece_cv:.4f}')
print(f'  Fixed test : {ece_test:.4f}')
print('='*60)


---
## Summary

| Stage | Output |
|---|---|
| Data | Patient-wise split + full dataset statistics + class distribution |
| Model | MAGNET: 4 backbones + CrossMagFusion + GNN + dual heads |
| Regularisation | Weight decay 1e-4, label smoothing ε=0.1, grad clip 1.0 |
| Training | AMP, cosine LR, early stopping, VRAM monitor |
| **Primary evaluation** | **5-fold patient-wise stratified CV with 95% CI** |
| Statistical tests | McNemar's exact test + proportion CI |
| Ablation | 5 configs (full / w/o GNN / w/o Aux / w/o Attn / w/o Both) |
| Baselines | 4 single-backbone + ConcatFusion (reviewer-requested) + MAGNET |
| SOTA table | Literature comparison with image/patient-split caveat |
| Interpretability | Attention heatmaps + Grad-CAM + MC-Dropout uncertainty |
| Exports | JSON, CSVs, 3 LaTeX tables, all plots |

**GTX 1650 notes:**
- Keep `BATCH_SIZE=4`. Peak VRAM ≈ 1150 MB (well within 4096 MB).
- Full run (training + 5-fold CV + baselines + ablation) ≈ 10–14 hours.
- Run cells 1–6 first to verify training works before committing to CV.